<a href="https://colab.research.google.com/github/brunosiqp/SENAI-projects/blob/main/Projeto_BD_Clinica_Veterinaria_PetVida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐾 Projeto de Banco de Dados — Clínica Veterinária **PetVida**
### Desafio — Modelagem e Armazenamento de Dados (525587)

| | |
|---|---|
| **Curso** | Pós-Graduação em Ciência de Dados e Inteligência Artificial — SENAI/SC (EAD) |
| **Unidade Curricular** | Modelagem e Armazenamento de Dados — 525587 |
| **Docentes** | Eric Tomas Zancanaro · Gustavo Stangherlin Cantarelli |
| **Grupo** | **Grupo P** |
| **SGBD** | PostgreSQL (instalado e executado dentro do próprio Colab) |
| **Segmento de mercado** | Saúde animal — clínica veterinária |
| **Prazo de entrega** | 29/11/2026 |

## 👥 Integrantes do grupo

| # | Nome |
|---|------|
| 1 | **Bruno Siqueira Parente** |

> Grupo composto por um único integrante.

## 📑 Sumário
1. Segmento de mercado e minimundo
2. Regras de negócio
3. Modelo lógico (diagrama + dicionário de dados)
4. Normalização (1FN, 2FN e 3FN)
5. Preparação do ambiente (instalação do PostgreSQL no Colab)
6. Implementação do banco (DDL com PKs, FKs e regras de integridade)
7. Carga dos dados (arquivos CSV e JSON + script de INSERTs)
8. Testes das regras de integridade
9. As 10 consultas SQL (com explicação)
10. Conclusão

**Como executar:** `Ambiente de execução ▸ Executar tudo` (Ctrl+F9). O notebook é autocontido: instala o PostgreSQL, cria o banco, gera os arquivos CSV/JSON, faz a carga e roda as consultas.

## 1. Segmento de mercado e minimundo

**Segmento escolhido:** clínicas veterinárias / pet care — um mercado em forte crescimento no Brasil, onde o controle de clientes (tutores), pacientes (animais), agenda médica e faturamento por serviço é essencial para decisões de negócio (quais serviços mais vendem, quais profissionais geram mais receita, quais clientes são mais fiéis etc.).

**Minimundo:** A clínica **PetVida** atende animais de estimação trazidos por seus **tutores**. Cada tutor mora em uma **cidade**, que pertence a um **estado**. Cada **animal** pertence a um único tutor e possui uma **raça**, que por sua vez pertence a uma **espécie** (cão, gato, ave, roedor). A clínica possui **veterinários**, cada um com uma **especialidade** principal. Um animal pode passar por vários **atendimentos**; cada atendimento é realizado por um veterinário em uma data/hora e possui um status (Agendado, Concluído ou Cancelado). Em um atendimento podem ser realizados vários **serviços** (consulta, vacina, exame, cirurgia…) e um mesmo serviço aparece em vários atendimentos — por isso existe a tabela associativa **atendimento_servico**, que guarda a quantidade e o valor efetivamente cobrado (que pode ter desconto em relação ao preço de tabela).

## 2. Regras de negócio

| Código | Regra | Como foi implementada |
|---|---|---|
| RN01 | Cada estado tem sigla única | `UNIQUE (sigla)` |
| RN02 | Uma cidade pertence a exatamente um estado; não existem duas cidades com o mesmo nome no mesmo estado | FK `cidade.id_estado` + `UNIQUE (nome, id_estado)` |
| RN03 | CPF e e-mail do tutor não podem se repetir | `UNIQUE (cpf)`, `UNIQUE (email)` |
| RN04 | Todo animal pertence a um tutor e a uma raça | FKs `NOT NULL` em `animal` |
| RN05 | Se um tutor for excluído, seus animais e respectivos atendimentos também são excluídos | `ON DELETE CASCADE` (tutor → animal → atendimento → atendimento_servico) |
| RN06 | Não é permitido excluir estado, cidade, raça, espécie, especialidade, veterinário ou serviço que esteja em uso | `ON DELETE RESTRICT` |
| RN07 | Alteração de chave primária é propagada | `ON UPDATE CASCADE` em todas as FKs |
| RN08 | Sexo do animal só pode ser `M` ou `F`; peso deve ser positivo | `CHECK` |
| RN09 | Status do atendimento: `Agendado`, `Concluído` ou `Cancelado` | `CHECK` + `DEFAULT 'Agendado'` |
| RN10 | Preço, quantidade e valor cobrado não podem ser negativos/zero | `CHECK` |
| RN11 | Um mesmo serviço não é lançado duas vezes no mesmo atendimento (usa-se a quantidade) | PK composta `(id_atendimento, id_servico)` |

## 3. Modelo lógico

Diagrama elaborado pela equipe (notação de tabelas relacionais com PK/FK e cardinalidade 1:N). São **11 tabelas**.

![Modelo lógico](attachment:modelo_logico.png)

### Relacionamentos

| Tabela pai (1) | Tabela filha (N) | FK | Ação ON DELETE |
|---|---|---|---|
| estado | cidade | `id_estado` | RESTRICT |
| cidade | tutor | `id_cidade` | RESTRICT |
| tutor | animal | `id_tutor` | CASCADE |
| especie | raca | `id_especie` | RESTRICT |
| raca | animal | `id_raca` | RESTRICT |
| especialidade | veterinario | `id_especialidade` | RESTRICT |
| animal | atendimento | `id_animal` | CASCADE |
| veterinario | atendimento | `id_veterinario` | RESTRICT |
| atendimento | atendimento_servico | `id_atendimento` | CASCADE |
| servico | atendimento_servico | `id_servico` | RESTRICT |

> O relacionamento **N:N** entre `atendimento` e `servico` foi resolvido com a tabela associativa `atendimento_servico`, cuja PK é composta pelas duas FKs.

### Dicionário de dados

| Tabela | Coluna | Tipo | Restrições | Descrição |
|---|---|---|---|---|
| **estado** | id_estado | INT | PK, identity | Identificador do estado |
| | sigla | CHAR(2) | NOT NULL, UNIQUE | UF (SC, PR…) |
| | nome | VARCHAR(50) | NOT NULL | Nome do estado |
| **cidade** | id_cidade | INT | PK | Identificador da cidade |
| | nome | VARCHAR(80) | NOT NULL | Nome da cidade |
| | id_estado | INT | FK → estado, NOT NULL | Estado da cidade |
| **tutor** | id_tutor | INT | PK | Identificador do tutor (cliente) |
| | nome | VARCHAR(100) | NOT NULL | Nome completo |
| | cpf | CHAR(14) | NOT NULL, UNIQUE | CPF formatado |
| | email | VARCHAR(120) | NOT NULL, UNIQUE | E-mail |
| | telefone | VARCHAR(20) | | Telefone |
| | data_cadastro | DATE | NOT NULL, DEFAULT CURRENT_DATE | Data de cadastro |
| | id_cidade | INT | FK → cidade, NOT NULL | Cidade onde mora |
| **especie** | id_especie | INT | PK | Identificador |
| | nome | VARCHAR(40) | NOT NULL, UNIQUE | Cão, Gato, Ave, Roedor |
| **raca** | id_raca | INT | PK | Identificador |
| | nome | VARCHAR(60) | NOT NULL | Nome da raça |
| | id_especie | INT | FK → especie, NOT NULL | Espécie da raça |
| **animal** | id_animal | INT | PK | Identificador do paciente |
| | nome | VARCHAR(60) | NOT NULL | Nome do animal |
| | data_nascimento | DATE | | Data de nascimento |
| | sexo | CHAR(1) | NOT NULL, CHECK (M/F) | Sexo |
| | peso_kg | NUMERIC(6,2) | CHECK (> 0) | Peso atual em kg |
| | id_tutor | INT | FK → tutor, NOT NULL | Tutor responsável |
| | id_raca | INT | FK → raca, NOT NULL | Raça |
| **especialidade** | id_especialidade | INT | PK | Identificador |
| | nome | VARCHAR(60) | NOT NULL, UNIQUE | Nome da especialidade |
| **veterinario** | id_veterinario | INT | PK | Identificador |
| | nome | VARCHAR(100) | NOT NULL | Nome do profissional |
| | crmv | VARCHAR(20) | NOT NULL, UNIQUE | Registro no CRMV |
| | id_especialidade | INT | FK → especialidade, NOT NULL | Especialidade principal |
| | data_admissao | DATE | NOT NULL | Data de admissão |
| **servico** | id_servico | INT | PK | Identificador |
| | descricao | VARCHAR(80) | NOT NULL, UNIQUE | Nome do serviço |
| | categoria | VARCHAR(30) | NOT NULL, CHECK | Consulta, Vacina, Exame, Cirurgia, Estética, Procedimento |
| | preco_base | NUMERIC(10,2) | NOT NULL, CHECK (≥ 0) | Preço de tabela |
| | ativo | BOOLEAN | NOT NULL, DEFAULT TRUE | Se o serviço ainda é oferecido |
| **atendimento** | id_atendimento | INT | PK | Identificador |
| | data_hora | TIMESTAMP | NOT NULL | Data e hora do atendimento |
| | id_animal | INT | FK → animal, NOT NULL | Paciente |
| | id_veterinario | INT | FK → veterinario, NOT NULL | Profissional responsável |
| | status | VARCHAR(12) | NOT NULL, CHECK, DEFAULT 'Agendado' | Agendado / Concluído / Cancelado |
| | observacao | VARCHAR(255) | | Observações clínicas |
| **atendimento_servico** | id_atendimento | INT | PK, FK → atendimento | Atendimento |
| | id_servico | INT | PK, FK → servico | Serviço realizado |
| | quantidade | INT | NOT NULL, CHECK (> 0) | Quantidade |
| | valor_unitario | NUMERIC(10,2) | NOT NULL, CHECK (≥ 0) | Valor efetivamente cobrado (com desconto) |

## 4. Normalização

O ponto de partida foi a **ficha de atendimento em papel** usada pela clínica, que concentrava tudo em um único registro (forma **não normalizada – 0FN**):

`FICHA (nº_atendimento, data_hora, nome_tutor, cpf_tutor, email, telefone, cidade, uf, nome_estado, nome_animal, sexo, peso, raça, espécie, nome_veterinário, crmv, especialidade, status, {serviço, categoria, preço_tabela, qtd, valor_cobrado})`

> `{ … }` indica um **grupo repetitivo** (vários serviços na mesma ficha).

### 1ª Forma Normal (1FN) — atributos atômicos, sem grupos repetitivos
* O grupo repetitivo de serviços foi retirado da ficha e virou uma relação própria, com chave composta: `ITEM (nº_atendimento, serviço, categoria, preço_tabela, qtd, valor_cobrado)`.
* Todos os atributos passaram a ser atômicos (ex.: telefone único por tutor, cidade e UF em colunas separadas).

### 2ª Forma Normal (2FN) — sem dependências parciais da chave composta
* Em `ITEM`, os atributos `categoria` e `preço_tabela` dependem **somente** de `serviço`, e não da chave inteira (`nº_atendimento + serviço`). Eles foram movidos para a tabela **`servico`**.
* Sobrou na associativa **`atendimento_servico`** apenas o que depende da chave completa: `quantidade` e `valor_unitario` (o valor cobrado naquele atendimento, que pode ter desconto — por isso **não** é redundante com `preco_base`).

### 3ª Forma Normal (3FN) — sem dependências transitivas
Na ficha, vários atributos dependiam de outro atributo não-chave:
| Dependência transitiva encontrada | Solução (nova tabela) |
|---|---|
| nº_atendimento → cpf_tutor → nome, e-mail, telefone, cidade | **tutor** |
| cidade → uf → nome_estado | **cidade** e **estado** |
| nº_atendimento → animal → sexo, peso, raça, tutor | **animal** |
| raça → espécie | **raca** e **especie** |
| nº_atendimento → crmv → nome_veterinário, especialidade | **veterinario** e **especialidade** |

Resultado: **11 tabelas em 3FN**, em que todo atributo não-chave depende **da chave, da chave inteira e de nada além da chave**. Valores derivados (ex.: total do atendimento = Σ quantidade × valor_unitario, idade do animal) **não** são armazenados — são calculados nas consultas, evitando anomalias de atualização.

## 5. Preparação do ambiente
Instala o servidor PostgreSQL na máquina virtual do Colab, inicia o serviço, define a senha do usuário `postgres` e cria o banco `petvida`. *(leva ~1 minuto)*

In [1]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y postgresql > /dev/null
!service postgresql start
!sudo -u postgres psql -q -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres psql -q -c "DROP DATABASE IF EXISTS petvida;"
!sudo -u postgres psql -q -c "CREATE DATABASE petvida ENCODING 'UTF8' TEMPLATE template0;"
!pip install -q psycopg2-binary

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[ OK ]
NOTICE:  database "petvida" does not exist, skipping


In [2]:
import psycopg2, pandas as pd, json, csv, os
from IPython.display import display

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

conn = psycopg2.connect(host='localhost', dbname='petvida', user='postgres', password='postgres')
conn.autocommit = True

def executar(sql):
    """Executa comandos DDL/DML (sem retorno de linhas)."""
    with conn.cursor() as cur:
        cur.execute(sql)

def consulta(sql):
    """Executa um SELECT e devolve o resultado como DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql)
        colunas = [c[0] for c in cur.description]
        df = pd.DataFrame(cur.fetchall(), columns=colunas)
    print(f'{len(df)} linha(s) retornada(s)')
    return df

with conn.cursor() as cur:
    cur.execute('SHOW server_version')
    print('Conectado ao PostgreSQL', cur.fetchone()[0])

Conectado ao PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)


## 6. Implementação do banco de dados (DDL)
Script de criação das 11 tabelas, com **PKs**, **FKs com regras de integridade referencial** (`ON DELETE CASCADE/RESTRICT`, `ON UPDATE CASCADE`) e **restrições de domínio** (`NOT NULL`, `UNIQUE`, `CHECK`, `DEFAULT`). O script é salvo como `schema.sql` e executado em seguida.

In [3]:
%%writefile schema.sql
-- =========================================================
-- Clínica Veterinária "PetVida" - Script DDL (PostgreSQL)
-- =========================================================
DROP TABLE IF EXISTS atendimento_servico, atendimento, servico, veterinario,
                     especialidade, animal, raca, especie, tutor, cidade, estado CASCADE;

CREATE TABLE estado (
    id_estado   INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    sigla       CHAR(2)      NOT NULL UNIQUE,
    nome        VARCHAR(50)  NOT NULL
);

CREATE TABLE cidade (
    id_cidade   INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    nome        VARCHAR(80)  NOT NULL,
    id_estado   INTEGER      NOT NULL,
    CONSTRAINT fk_cidade_estado FOREIGN KEY (id_estado)
        REFERENCES estado (id_estado) ON UPDATE CASCADE ON DELETE RESTRICT,
    CONSTRAINT uq_cidade_estado UNIQUE (nome, id_estado)
);

CREATE TABLE tutor (
    id_tutor      INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    nome          VARCHAR(100) NOT NULL,
    cpf           CHAR(14)     NOT NULL UNIQUE,
    email         VARCHAR(120) NOT NULL UNIQUE,
    telefone      VARCHAR(20),
    data_cadastro DATE         NOT NULL DEFAULT CURRENT_DATE,
    id_cidade     INTEGER      NOT NULL,
    CONSTRAINT fk_tutor_cidade FOREIGN KEY (id_cidade)
        REFERENCES cidade (id_cidade) ON UPDATE CASCADE ON DELETE RESTRICT
);

CREATE TABLE especie (
    id_especie  INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    nome        VARCHAR(40)  NOT NULL UNIQUE
);

CREATE TABLE raca (
    id_raca     INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    nome        VARCHAR(60)  NOT NULL,
    id_especie  INTEGER      NOT NULL,
    CONSTRAINT fk_raca_especie FOREIGN KEY (id_especie)
        REFERENCES especie (id_especie) ON UPDATE CASCADE ON DELETE RESTRICT,
    CONSTRAINT uq_raca_especie UNIQUE (nome, id_especie)
);

CREATE TABLE animal (
    id_animal       INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    nome            VARCHAR(60)   NOT NULL,
    data_nascimento DATE,
    sexo            CHAR(1)       NOT NULL CHECK (sexo IN ('M', 'F')),
    peso_kg         NUMERIC(6,2)  CHECK (peso_kg > 0),
    id_tutor        INTEGER       NOT NULL,
    id_raca         INTEGER       NOT NULL,
    CONSTRAINT fk_animal_tutor FOREIGN KEY (id_tutor)
        REFERENCES tutor (id_tutor) ON UPDATE CASCADE ON DELETE CASCADE,
    CONSTRAINT fk_animal_raca FOREIGN KEY (id_raca)
        REFERENCES raca (id_raca) ON UPDATE CASCADE ON DELETE RESTRICT
);

CREATE TABLE especialidade (
    id_especialidade INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    nome             VARCHAR(60) NOT NULL UNIQUE
);

CREATE TABLE veterinario (
    id_veterinario   INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    nome             VARCHAR(100) NOT NULL,
    crmv             VARCHAR(20)  NOT NULL UNIQUE,
    id_especialidade INTEGER      NOT NULL,
    data_admissao    DATE         NOT NULL,
    CONSTRAINT fk_vet_especialidade FOREIGN KEY (id_especialidade)
        REFERENCES especialidade (id_especialidade) ON UPDATE CASCADE ON DELETE RESTRICT
);

CREATE TABLE servico (
    id_servico  INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    descricao   VARCHAR(80)   NOT NULL UNIQUE,
    categoria   VARCHAR(30)   NOT NULL
                CHECK (categoria IN ('Consulta','Vacina','Exame','Cirurgia','Estética','Procedimento')),
    preco_base  NUMERIC(10,2) NOT NULL CHECK (preco_base >= 0),
    ativo       BOOLEAN       NOT NULL DEFAULT TRUE
);

CREATE TABLE atendimento (
    id_atendimento INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    data_hora      TIMESTAMP    NOT NULL,
    id_animal      INTEGER      NOT NULL,
    id_veterinario INTEGER      NOT NULL,
    status         VARCHAR(12)  NOT NULL DEFAULT 'Agendado'
                   CHECK (status IN ('Agendado','Concluído','Cancelado')),
    observacao     VARCHAR(255),
    CONSTRAINT fk_atend_animal FOREIGN KEY (id_animal)
        REFERENCES animal (id_animal) ON UPDATE CASCADE ON DELETE CASCADE,
    CONSTRAINT fk_atend_vet FOREIGN KEY (id_veterinario)
        REFERENCES veterinario (id_veterinario) ON UPDATE CASCADE ON DELETE RESTRICT
);

CREATE TABLE atendimento_servico (
    id_atendimento INTEGER       NOT NULL,
    id_servico     INTEGER       NOT NULL,
    quantidade     INTEGER       NOT NULL DEFAULT 1 CHECK (quantidade > 0),
    valor_unitario NUMERIC(10,2) NOT NULL CHECK (valor_unitario >= 0),
    CONSTRAINT pk_atendimento_servico PRIMARY KEY (id_atendimento, id_servico),
    CONSTRAINT fk_as_atendimento FOREIGN KEY (id_atendimento)
        REFERENCES atendimento (id_atendimento) ON UPDATE CASCADE ON DELETE CASCADE,
    CONSTRAINT fk_as_servico FOREIGN KEY (id_servico)
        REFERENCES servico (id_servico) ON UPDATE CASCADE ON DELETE RESTRICT
);

Writing schema.sql


In [4]:
executar(open('schema.sql', encoding='utf-8').read())
print('Tabelas criadas com sucesso!')

Tabelas criadas com sucesso!


Conferência das restrições criadas (PK, FK, UNIQUE e CHECK) a partir do catálogo do PostgreSQL:

In [5]:
consulta("""
SELECT tc.table_name      AS tabela,
       tc.constraint_type AS tipo,
       tc.constraint_name AS restricao
FROM information_schema.table_constraints tc
WHERE tc.table_schema = 'public'
  AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY', 'UNIQUE', 'CHECK')
  AND tc.constraint_name NOT LIKE '%_not_null'
ORDER BY tc.table_name, tc.constraint_type;
""")

37 linha(s) retornada(s)


,tabela,tipo,restricao
0,animal,CHECK,animal_sexo_check
1,animal,CHECK,animal_peso_kg_check
2,animal,FOREIGN KEY,fk_animal_tutor
3,animal,FOREIGN KEY,fk_animal_raca
4,animal,PRIMARY KEY,animal_pkey
5,atendimento,CHECK,atendimento_status_check
6,atendimento,FOREIGN KEY,fk_atend_vet
7,atendimento,FOREIGN KEY,fk_atend_animal
8,atendimento,PRIMARY KEY,atendimento_pkey
9,atendimento_servico,CHECK,atendimento_servico_valor_unitario_check


## 7. Carga dos dados

### 🔗 Arquivos de dados (CSV e JSON) e script de INSERTs
Todos os arquivos usados na carga estão **incorporados neste notebook**: as células da seção 7.1 criam cada arquivo na pasta `dados/` do Colab (`%%writefile`), e a seção 7.3 traz o **script completo de `INSERT`s** (`inserts.sql`). Depois de executar, os arquivos podem ser abertos pelo ícone de pasta 📁 na barra lateral do Colab.

| Arquivo | Formato | Tabela | Registros |
|---|---|---|---|
| `estado.csv` | CSV | estado | 5 |
| `cidade.csv` | CSV | cidade | 12 |
| `tutor.csv` | CSV | tutor | 40 |
| `especie.csv` | CSV | especie | 4 |
| `raca.csv` | CSV | raca | 14 |
| `animal.csv` | CSV | animal | 60 |
| `especialidade.csv` | CSV | especialidade | 6 |
| `veterinario.csv` | CSV | veterinario | 8 |
| `servico.json` | JSON | servico | 15 |
| `atendimento.csv` | CSV | atendimento | 160 |
| `atendimento_servico.json` | JSON | atendimento_servico | 273 |
| `inserts.sql` | SQL | todas (carga alternativa via INSERT) | — |

Para que o notebook rode sozinho, as células abaixo **recriam os mesmos arquivos** na pasta `dados/` do Colab (comando `%%writefile`). Em seguida, os **CSVs são importados com `COPY`** e os **JSONs são lidos com Python e inseridos com `INSERT`**.

### 7.1 Arquivos de dados

In [6]:
os.makedirs('dados', exist_ok=True)

In [7]:
%%writefile dados/estado.csv
id_estado,sigla,nome
1,SC,Santa Catarina
2,PR,Paraná
3,RS,Rio Grande do Sul
4,SP,São Paulo
5,RJ,Rio de Janeiro

Writing dados/estado.csv


In [8]:
%%writefile dados/cidade.csv
id_cidade,nome,id_estado
1,Blumenau,1
2,Florianópolis,1
3,Joinville,1
4,Itajaí,1
5,Curitiba,2
6,Londrina,2
7,Porto Alegre,3
8,Caxias do Sul,3
9,São Paulo,4
10,Campinas,4
11,Rio de Janeiro,5
12,Niterói,5

Writing dados/cidade.csv


In [9]:
%%writefile dados/tutor.csv
id_tutor,nome,cpf,email,telefone,data_cadastro,id_cidade
1,Ana Santos,242.854.204-11,ana.santos1@gmail.com,(21) 95506-5012,2024-08-16,11
2,Bruno Schmidt,195.323.338-12,bruno.schmidt2@gmail.com,(11) 97912-1520,2024-01-31,9
3,Carla Barbosa,325.559.703-13,carla.barbosa3@gmail.com,(11) 94257-9928,2025-03-05,5
4,Daniel Silva,259.320.881-14,daniel.silva4@hotmail.com,(21) 97924-6574,2024-10-11,6
5,Eduarda Santos,452.718.370-15,eduarda.santos5@gmail.com,(51) 92584-6881,2026-05-17,1
6,Felipe Martins,400.949.743-16,felipe.martins6@gmail.com,(47) 97201-2291,2025-07-19,10
7,Gabriela Ferreira,333.891.396-17,gabriela.ferreira7@gmail.com,(48) 92139-1750,2025-11-08,2
8,Henrique Almeida,954.473.266-18,henrique.almeida8@gmail.com,(51) 95554-8428,2025-10-12,6
9,Isabela Ferreira,750.275.646-19,isabela.ferreira9@hotmail.com,(21) 95374-2169,2025-09-15,12
10,João Almeida,804.670.324-20,joao.almeida10@hotmail.com,(51) 97216-5422,2025-10-17,11
11,Larissa Araújo,374.167.316-21,larissa.araujo11@gmail.com,(48) 91525-6168,2025-02-14,10
12,Marcos Araújo,569.246.371-22,marcos.araujo12@hotmail.com,(21) 99179-7482,2025-10-20,3
13,Mariana Almeida,508.470.324-23,mariana.almeida13@gmail.com,(11) 95304-8019,2025-08-20,3
14,Natália Ribeiro,742.263.911-24,natalia.ribeiro14@yahoo.com.br,(47) 91771-2796,2024-06-05,11
15,Otávio Gomes,579.641.357-25,otavio.gomes15@gmail.com,(47) 97304-7252,2025-09-02,9
16,Paula Silva,756.448.214-26,paula.silva16@gmail.com,(21) 99797-5371,2026-02-26,5
17,Rafael Gomes,880.282.619-27,rafael.gomes17@hotmail.com,(51) 91053-5315,2025-05-27,2
18,Renata Lima,880.265.652-28,renata.lima18@gmail.com,(11) 94258-3504,2025-01-17,9
19,Sérgio Silva,471.999.951-29,sergio.silva19@gmail.com,(41) 99005-1319,2024-04-24,5
20,Tatiane Almeida,597.935.170-30,tatiane.almeida20@gmail.com,(48) 92290-2403,2026-01-19,9
21,Lucas Pereira,640.993.721-31,lucas.pereira21@hotmail.com,(21) 98787-3705,2024-09-28,7
22,Juliana Rodrigues,787.765.482-32,juliana.rodrigues22@gmail.com,(21) 94295-6107,2025-02-12,8
23,Marcelo Ribeiro,446.121.702-33,marcelo.ribeiro23@yahoo.com.br,(47) 95061-4681,2024-03-06,9
24,Fernanda Almeida,746.160.334-34,fernanda.almeida24@gmail.com,(48) 91117-2163,2025-12-25,2
25,Gustavo Souza,785.597.319-35,gustavo.souza25@outlook.com,(47) 99423-4899,2024-10-12,9
26,Camila Pereira,584.926.516-36,camila.pereira26@gmail.com,(11) 98744-4981,2026-03-14,4
27,Rodrigo Santos,520.578.984-37,rodrigo.santos27@gmail.com,(21) 98062-6804,2025-03-09,12
28,Patrícia Souza,982.211.354-38,patricia.souza28@gmail.com,(47) 97596-6559,2026-03-30,4
29,Thiago Rodrigues,385.573.355-39,thiago.rodrigues29@gmail.com,(51) 93296-7912,2024-07-06,2
30,Beatriz Martins,115.195.871-40,beatriz.martins30@gmail.com,(47) 91828-9856,2026-05-06,4
31,Leonardo Costa,510.160.268-41,leonardo.costa31@yahoo.com.br,(51) 98886-4502,2026-06-04,7
32,Amanda Silva,813.848.902-42,amanda.silva32@yahoo.com.br,(41) 98454-5673,2025-03-09,9
33,Vinícius Rocha,693.853.655-43,vinicius.rocha33@hotmail.com,(48) 95861-4566,2024-02-29,1
34,Letícia Araújo,643.261.158-44,leticia.araujo34@gmail.com,(47) 98811-9238,2026-05-23,9
35,André Oliveira,222.683.352-45,andre.oliveira35@hotmail.com,(47) 92113-4853,2025-02-17,10
36,Aline Barbosa,697.678.635-46,aline.barbosa36@gmail.com,(11) 92343-7868,2025-11-04,6
37,Diego Nascimento,505.234.787-47,diego.nascimento37@hotmail.com,(21) 96147-4910,2024-09-28,11
38,Bianca Lima,736.676.202-48,bianca.lima38@yahoo.com.br,(41) 92188-1152,2025-04-14,2
39,Matheus Schmidt,170.350.478-49,matheus.schmidt39@hotmail.com,(11) 95345-3170,2024-12-23,5
40,Priscila Costa,783.936.667-50,priscila.costa40@yahoo.com.br,(11) 95956-9666,2024-01-09,5

Writing dados/tutor.csv


In [10]:
%%writefile dados/especie.csv
id_especie,nome
1,Cão
2,Gato
3,Ave
4,Roedor

Writing dados/especie.csv


In [11]:
%%writefile dados/raca.csv
id_raca,nome,id_especie
1,Labrador Retriever,1
2,Golden Retriever,1
3,Shih Tzu,1
4,Bulldog Francês,1
5,Pastor Alemão,1
6,SRD (Cão),1
7,Siamês,2
8,Persa,2
9,Maine Coon,2
10,SRD (Gato),2
11,Calopsita,3
12,Periquito Australiano,3
13,Hamster Sírio,4
14,Porquinho-da-índia,4

Writing dados/raca.csv


In [12]:
%%writefile dados/animal.csv
id_animal,nome,data_nascimento,sexo,peso_kg,id_tutor,id_raca
1,Thor,2014-04-28,M,0.04,1,12
2,Mel,2024-05-29,M,0.09,2,11
3,Luna,2019-09-09,M,5.06,3,7
4,Bob,2023-05-03,F,4.0,4,10
5,Nina,2013-02-20,M,0.11,5,11
6,Pipoca,2012-12-27,M,4.33,6,10
7,Max,2017-11-15,M,34.64,7,2
8,Bela,2021-08-04,M,7.8,8,6
9,Fred,2015-05-06,M,0.18,9,13
10,Lola,2015-04-28,F,8.19,10,6
11,Toby,2012-11-22,F,5.05,11,3
12,Amora,2014-04-22,F,7.9,12,3
13,Simba,2021-02-11,M,0.12,13,11
14,Frida,2015-08-24,M,8.41,14,3
15,Zeus,2019-06-14,F,37.03,15,1
16,Pandora,2017-07-25,F,3.64,16,10
17,Paçoca,2020-07-30,M,5.58,17,8
18,Kiara,2022-04-28,F,5.53,18,3
19,Bidu,2017-02-07,M,3.09,19,10
20,Cacau,2019-05-13,F,36.24,20,2
21,Romeu,2018-04-05,F,0.16,21,13
22,Julieta,2024-01-10,F,43.97,22,5
23,Fumaça,2017-11-09,M,32.55,23,2
24,Mia,2012-11-09,M,1.0,24,14
25,Rex,2019-01-13,F,11.64,25,4
26,Jade,2020-08-21,M,11.37,26,6
27,Chico,2012-01-14,M,4.09,27,8
28,Belinha,2019-05-28,F,34.95,28,1
29,Ozzy,2018-09-26,F,33.67,29,2
30,Maya,2018-08-19,M,9.15,30,4
31,Pingo,2020-07-02,M,4.85,31,8
32,Sofia,2024-04-15,M,5.52,32,3
33,Duke,2025-01-03,F,6.32,33,3
34,Estrela,2016-10-16,F,41.49,34,5
35,Tobias,2015-10-22,M,0.13,35,13
36,Lua,2019-07-08,M,5.46,36,8
37,Nala,2018-12-17,M,9.03,37,9
38,Pretinha,2013-01-13,M,37.79,38,2
39,Floquinho,2013-08-19,F,4.24,39,7
40,Bolinha,2016-05-11,F,4.48,40,7
41,Margarida,2012-02-15,M,7.89,15,3
42,Apolo,2023-08-14,F,25.65,16,2
43,Safira,2014-09-21,F,0.03,30,12
44,Sansão,2024-07-14,F,5.85,29,8
45,Lili,2023-04-28,F,5.49,36,7
46,Bento,2015-07-27,F,16.25,16,6
47,Pérola,2018-03-21,F,5.51,18,10
48,Marley,2018-05-29,M,32.62,21,5
49,Tieta,2013-10-22,M,0.09,25,11
50,Kiko,2016-10-18,M,4.24,22,8
51,Dora,2021-04-29,M,10.17,27,6
52,Hulk,2025-02-05,M,42.56,37,5
53,Pituca,2012-02-18,F,9.79,25,4
54,Billy,2021-05-25,M,4.95,18,10
55,Princesa,2012-08-25,F,33.71,26,5
56,Teodoro,2022-06-25,M,9.9,35,9
57,Laila,2020-11-01,M,26.26,28,1
58,Gaia,2022-05-10,M,25.65,25,2
59,Brisa,2022-03-13,F,10.03,25,4
60,Faísca,2021-06-15,F,8.17,31,3

Writing dados/animal.csv


In [13]:
%%writefile dados/especialidade.csv
id_especialidade,nome
1,Clínica Geral
2,Dermatologia
3,Cardiologia
4,Ortopedia
5,Animais Silvestres e Exóticos
6,Odontologia

Writing dados/especialidade.csv


In [14]:
%%writefile dados/veterinario.csv
id_veterinario,nome,crmv,id_especialidade,data_admissao
1,Dra. Helena Duarte,CRMV-SC 10234,1,2018-03-01
2,Dr. Ricardo Mendes,CRMV-SC 11876,4,2019-07-15
3,Dra. Sabrina Kowalski,CRMV-PR 08451,2,2020-02-10
4,Dr. Fábio Teixeira,CRMV-SC 12990,3,2021-05-03
5,Dra. Lívia Moraes,CRMV-SC 13402,5,2022-01-20
6,Dr. Caio Batista,CRMV-RS 07765,6,2022-09-12
7,Dra. Viviane Lopes,CRMV-SC 14051,1,2023-04-01
8,Dr. Eduardo Prado,CRMV-SP 21456,1,2024-08-19

Writing dados/veterinario.csv


In [15]:
%%writefile dados/servico.json
[
  {"id_servico": 1, "descricao": "Consulta clínica", "categoria": "Consulta", "preco_base": 150.0, "ativo": true},
  {"id_servico": 2, "descricao": "Consulta de retorno", "categoria": "Consulta", "preco_base": 80.0, "ativo": true},
  {"id_servico": 3, "descricao": "Consulta especializada", "categoria": "Consulta", "preco_base": 220.0, "ativo": true},
  {"id_servico": 4, "descricao": "Vacina V10 (cães)", "categoria": "Vacina", "preco_base": 95.0, "ativo": true},
  {"id_servico": 5, "descricao": "Vacina antirrábica", "categoria": "Vacina", "preco_base": 60.0, "ativo": true},
  {"id_servico": 6, "descricao": "Vacina V4 felina", "categoria": "Vacina", "preco_base": 110.0, "ativo": true},
  {"id_servico": 7, "descricao": "Hemograma completo", "categoria": "Exame", "preco_base": 75.0, "ativo": true},
  {"id_servico": 8, "descricao": "Ultrassonografia abdominal", "categoria": "Exame", "preco_base": 230.0, "ativo": true},
  {"id_servico": 9, "descricao": "Raio-X", "categoria": "Exame", "preco_base": 180.0, "ativo": true},
  {"id_servico": 10, "descricao": "Eletrocardiograma", "categoria": "Exame", "preco_base": 160.0, "ativo": true},
  {"id_servico": 11, "descricao": "Castração", "categoria": "Cirurgia", "preco_base": 650.0, "ativo": true},
  {"id_servico": 12, "descricao": "Limpeza de tártaro", "categoria": "Cirurgia", "preco_base": 420.0, "ativo": true},
  {"id_servico": 13, "descricao": "Banho e tosa", "categoria": "Estética", "preco_base": 90.0, "ativo": true},
  {"id_servico": 14, "descricao": "Aplicação de medicamento", "categoria": "Procedimento", "preco_base": 35.0, "ativo": true},
  {"id_servico": 15, "descricao": "Microchipagem", "categoria": "Procedimento", "preco_base": 120.0, "ativo": false}
]

Writing dados/servico.json


In [16]:
%%writefile dados/atendimento.csv
id_atendimento,data_hora,id_animal,id_veterinario,status,observacao
1,2026-01-09 15:30:00,44,6,Concluído,Prescrito antibiótico
2,2026-01-14 16:00:00,23,1,Concluído,Acompanhamento semestral
3,2026-01-20 08:30:00,55,3,Concluído,Sem alterações
4,2026-01-23 12:00:00,3,7,Cancelado,Cancelado pelo tutor
5,2026-01-26 12:30:00,3,6,Concluído,Animal estável
6,2026-01-26 13:30:00,18,7,Concluído,Animal estável
7,2026-01-30 09:30:00,29,7,Concluído,Sem alterações
8,2026-02-02 09:00:00,20,1,Concluído,Acompanhamento semestral
9,2026-02-04 09:00:00,35,5,Concluído,Animal estável
10,2026-02-05 15:00:00,52,8,Concluído,Acompanhamento semestral
11,2026-02-09 11:30:00,24,5,Concluído,Dieta recomendada
12,2026-02-09 15:30:00,41,2,Concluído,Animal estável
13,2026-02-10 14:00:00,28,1,Cancelado,Cancelado pelo tutor
14,2026-02-11 17:00:00,10,1,Concluído,Prescrito antibiótico
15,2026-02-12 10:00:00,56,7,Concluído,Sem alterações
16,2026-02-14 08:30:00,14,8,Concluído,Acompanhamento semestral
17,2026-02-14 13:30:00,42,1,Concluído,Retorno em 30 dias
18,2026-02-18 12:30:00,16,7,Concluído,Acompanhamento semestral
19,2026-02-19 14:00:00,16,4,Concluído,Animal estável
20,2026-02-23 10:00:00,31,2,Concluído,Prescrito antibiótico
21,2026-02-23 13:30:00,21,5,Concluído,Prescrito antibiótico
22,2026-02-23 16:30:00,19,1,Concluído,Dieta recomendada
23,2026-02-25 14:30:00,46,4,Concluído,Retorno em 30 dias
24,2026-02-25 15:00:00,45,2,Concluído,Prescrito antibiótico
25,2026-02-25 16:00:00,43,5,Concluído,Dieta recomendada
26,2026-02-27 10:30:00,27,7,Concluído,Sem alterações
27,2026-02-28 14:30:00,41,8,Concluído,Retorno em 30 dias
28,2026-03-02 14:30:00,51,8,Concluído,Acompanhamento semestral
29,2026-03-04 16:00:00,42,1,Concluído,Retorno em 30 dias
30,2026-03-06 16:30:00,39,6,Concluído,Dieta recomendada
31,2026-03-07 08:30:00,48,3,Concluído,Dieta recomendada
32,2026-03-09 11:00:00,16,7,Concluído,Dieta recomendada
33,2026-03-09 12:00:00,49,5,Concluído,Sem alterações
34,2026-03-10 13:00:00,29,4,Concluído,Retorno em 30 dias
35,2026-03-12 13:00:00,21,5,Concluído,Animal estável
36,2026-03-18 09:00:00,19,8,Concluído,Sem alterações
37,2026-03-20 10:30:00,52,1,Concluído,Animal estável
38,2026-03-21 11:00:00,10,1,Concluído,Retorno em 30 dias
39,2026-03-23 15:00:00,9,5,Concluído,Dieta recomendada
40,2026-03-24 10:00:00,54,7,Concluído,Retorno em 30 dias
41,2026-03-26 08:30:00,29,7,Concluído,Animal estável
42,2026-03-26 14:30:00,31,7,Concluído,Prescrito antibiótico
43,2026-03-28 11:00:00,56,1,Concluído,Prescrito antibiótico
44,2026-04-03 08:00:00,46,4,Concluído,Dieta recomendada
45,2026-04-07 10:30:00,5,5,Concluído,Acompanhamento semestral
46,2026-04-11 11:30:00,15,8,Concluído,Prescrito antibiótico
47,2026-04-13 09:30:00,6,7,Concluído,Dieta recomendada
48,2026-04-13 12:30:00,56,7,Concluído,Prescrito antibiótico
49,2026-04-16 09:00:00,7,7,Concluído,Acompanhamento semestral
50,2026-04-16 14:30:00,7,1,Concluído,Dieta recomendada
51,2026-04-20 09:00:00,38,1,Concluído,Prescrito antibiótico
52,2026-04-20 14:30:00,56,8,Concluído,Acompanhamento semestral
53,2026-04-21 15:30:00,31,3,Cancelado,Cancelado pelo tutor
54,2026-04-24 13:30:00,13,5,Concluído,Retorno em 30 dias
55,2026-04-27 11:30:00,44,7,Concluído,Acompanhamento semestral
56,2026-04-27 14:00:00,1,5,Concluído,Retorno em 30 dias
57,2026-04-28 09:30:00,59,7,Concluído,Animal estável
58,2026-04-28 17:00:00,47,7,Concluído,Dieta recomendada
59,2026-04-29 09:30:00,55,6,Concluído,Prescrito antibiótico
60,2026-05-06 12:00:00,4,1,Concluído,Dieta recomendada
61,2026-05-06 15:30:00,49,5,Concluído,Sem alterações
62,2026-05-06 17:30:00,2,5,Concluído,Animal estável
63,2026-05-08 15:30:00,29,7,Concluído,Retorno em 30 dias
64,2026-05-09 08:00:00,13,5,Concluído,Prescrito antibiótico
65,2026-05-11 11:00:00,40,1,Concluído,Acompanhamento semestral
66,2026-05-13 11:00:00,47,1,Concluído,Dieta recomendada
67,2026-05-14 13:00:00,50,3,Concluído,Animal estável
68,2026-05-20 17:00:00,47,1,Concluído,Dieta recomendada
69,2026-05-20 17:30:00,13,5,Cancelado,Cancelado pelo tutor
70,2026-05-21 09:30:00,6,3,Cancelado,Cancelado pelo tutor
71,2026-05-25 10:00:00,40,7,Concluído,Animal estável
72,2026-05-26 16:00:00,34,8,Concluído,Acompanhamento semestral
73,2026-05-27 11:00:00,2,5,Concluído,Sem alterações
74,2026-05-29 15:00:00,44,1,Concluído,Acompanhamento semestral
75,2026-05-30 09:30:00,13,5,Concluído,Dieta recomendada
76,2026-05-30 15:30:00,29,2,Cancelado,Cancelado pelo tutor
77,2026-05-30 15:30:00,57,8,Concluído,Sem alterações
78,2026-06-02 09:30:00,2,5,Cancelado,Cancelado pelo tutor
79,2026-06-05 08:30:00,18,1,Concluído,Acompanhamento semestral
80,2026-06-06 17:30:00,20,6,Concluído,Acompanhamento semestral
81,2026-06-09 15:00:00,7,1,Concluído,Sem alterações
82,2026-06-09 17:00:00,33,7,Concluído,Dieta recomendada
83,2026-06-10 17:30:00,31,6,Concluído,Acompanhamento semestral
84,2026-06-11 17:30:00,29,1,Concluído,Acompanhamento semestral
85,2026-06-12 15:00:00,24,5,Concluído,Dieta recomendada
86,2026-06-12 15:30:00,3,1,Concluído,Animal estável
87,2026-06-13 09:00:00,60,2,Concluído,Sem alterações
88,2026-06-18 17:00:00,5,5,Concluído,Dieta recomendada
89,2026-06-22 10:00:00,42,7,Concluído,Animal estável
90,2026-06-22 12:30:00,29,6,Concluído,Acompanhamento semestral
91,2026-06-22 17:30:00,13,5,Concluído,Sem alterações
92,2026-06-23 10:30:00,42,1,Concluído,Acompanhamento semestral
93,2026-06-24 15:30:00,10,6,Concluído,Prescrito antibiótico
94,2026-06-25 10:00:00,3,2,Concluído,Acompanhamento semestral
95,2026-06-27 10:30:00,45,4,Concluído,Retorno em 30 dias
96,2026-06-27 11:30:00,50,1,Concluído,Animal estável
97,2026-06-30 11:00:00,50,6,Concluído,Retorno em 30 dias
98,2026-07-01 16:30:00,43,5,Concluído,Dieta recomendada
99,2026-07-02 08:30:00,4,3,Concluído,Sem alterações
100,2026-07-03 12:30:00,9,5,Concluído,Animal estável
101,2026-07-06 14:00:00,19,6,Concluído,Dieta recomendada
102,2026-07-11 16:30:00,54,1,Cancelado,Cancelado pelo tutor
103,2026-07-13 11:30:00,29,8,Concluído,Animal estável
104,2026-07-13 12:30:00,50,8,Concluído,Animal estável
105,2026-07-14 09:00:00,31,1,Cancelado,Cancelado pelo tutor
106,2026-07-22 14:30:00,44,6,Concluído,Acompanhamento semestral
107,2026-07-25 16:30:00,16,1,Concluído,Animal estável
108,2026-07-27 09:30:00,55,7,Concluído,Prescrito antibiótico
109,2026-07-29 08:00:00,24,5,Concluído,Animal estável
110,2026-07-31 12:30:00,5,5,Concluído,Retorno em 30 dias
111,2026-07-31 13:00:00,2,5,Concluído,Dieta recomendada
112,2026-08-06 10:00:00,34,8,Concluído,Dieta recomendada
113,2026-08-13 10:00:00,34,3,Concluído,Retorno em 30 dias
114,2026-08-14 09:30:00,52,1,Concluído,Dieta recomendada
115,2026-08-17 14:30:00,58,8,Concluído,Retorno em 30 dias
116,2026-08-18 09:30:00,1,5,Concluído,Sem alterações
117,2026-08-19 12:00:00,38,4,Concluído,Animal estável
118,2026-08-21 11:30:00,45,8,Concluído,Animal estável
119,2026-08-24 10:30:00,6,7,Cancelado,Cancelado pelo tutor
120,2026-08-24 13:30:00,29,7,Concluído,Dieta recomendada
121,2026-08-31 11:30:00,39,1,Concluído,Prescrito antibiótico
122,2026-08-31 12:30:00,11,6,Cancelado,Cancelado pelo tutor
123,2026-09-02 08:30:00,24,5,Concluído,Animal estável
124,2026-09-02 12:00:00,55,1,Concluído,Prescrito antibiótico
125,2026-09-09 12:30:00,7,8,Concluído,Prescrito antibiótico
126,2026-09-10 09:00:00,41,7,Concluído,Retorno em 30 dias
127,2026-09-14 15:30:00,44,8,Concluído,Acompanhamento semestral
128,2026-09-15 10:30:00,57,1,Concluído,Dieta recomendada
129,2026-09-15 13:00:00,32,1,Concluído,Prescrito antibiótico
130,2026-09-18 16:30:00,29,1,Concluído,Sem alterações
131,2026-09-22 10:00:00,18,7,Concluído,Prescrito antibiótico
132,2026-09-22 12:30:00,54,4,Concluído,Sem alterações
133,2026-09-24 09:30:00,28,1,Concluído,Dieta recomendada
134,2026-09-28 13:30:00,41,4,Concluído,Prescrito antibiótico
135,2026-09-29 12:30:00,30,4,Concluído,Prescrito antibiótico
136,2026-10-01 09:00:00,31,4,Concluído,Dieta recomendada
137,2026-10-05 10:30:00,3,8,Concluído,Acompanhamento semestral
138,2026-10-06 08:00:00,46,1,Concluído,Dieta recomendada
139,2026-10-06 09:30:00,24,5,Concluído,Acompanhamento semestral
140,2026-10-08 14:00:00,57,3,Concluído,Sem alterações
141,2026-10-09 12:30:00,55,1,Concluído,Acompanhamento semestral
142,2026-10-13 12:00:00,9,5,Cancelado,Cancelado pelo tutor
143,2026-10-14 11:30:00,29,8,Concluído,Sem alterações
144,2026-10-14 14:00:00,15,8,Cancelado,Cancelado pelo tutor
145,2026-10-17 08:30:00,2,5,Concluído,Acompanhamento semestral
146,2026-10-17 13:00:00,13,5,Concluído,Sem alterações
147,2026-10-19 13:30:00,34,3,Cancelado,Cancelado pelo tutor
148,2026-10-22 11:00:00,40,8,Cancelado,Cancelado pelo tutor
149,2026-10-26 16:30:00,58,8,Concluído,Retorno em 30 dias
150,2026-10-26 17:00:00,49,5,Concluído,Prescrito antibiótico
151,2026-10-27 12:00:00,50,6,Concluído,Animal estável
152,2026-10-28 15:00:00,29,5,Concluído,Dieta recomendada
153,2026-11-04 13:30:00,46,6,Concluído,Animal estável
154,2026-11-05 16:30:00,30,6,Cancelado,Cancelado pelo tutor
155,2026-11-06 16:30:00,30,4,Concluído,Acompanhamento semestral
156,2026-11-13 14:00:00,32,8,Concluído,Sem alterações
157,2026-11-14 09:00:00,20,3,Concluído,Sem alterações
158,2026-11-17 09:00:00,41,7,Concluído,Retorno em 30 dias
159,2026-11-20 17:30:00,54,8,Concluído,Animal estável
160,2026-11-21 11:30:00,43,5,Concluído,Retorno em 30 dias

Writing dados/atendimento.csv


In [17]:
%%writefile dados/atendimento_servico.json
[
  {"id_atendimento": 1, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 1, "id_servico": 12, "quantidade": 1, "valor_unitario": 420.0},
  {"id_atendimento": 1, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 2, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 3, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 3, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 4, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 4, "id_servico": 7, "quantidade": 1, "valor_unitario": 67.5},
  {"id_atendimento": 4, "id_servico": 14, "quantidade": 2, "valor_unitario": 35.0},
  {"id_atendimento": 5, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 5, "id_servico": 12, "quantidade": 1, "valor_unitario": 399.0},
  {"id_atendimento": 5, "id_servico": 14, "quantidade": 2, "valor_unitario": 35.0},
  {"id_atendimento": 6, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 7, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 8, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 9, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 9, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 9, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 10, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 11, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 12, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 13, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 13, "id_servico": 14, "quantidade": 1, "valor_unitario": 31.5},
  {"id_atendimento": 14, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 15, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 15, "id_servico": 6, "quantidade": 1, "valor_unitario": 110.0},
  {"id_atendimento": 15, "id_servico": 14, "quantidade": 2, "valor_unitario": 35.0},
  {"id_atendimento": 16, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 16, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 17, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 17, "id_servico": 5, "quantidade": 1, "valor_unitario": 54.0},
  {"id_atendimento": 18, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 19, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 20, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 21, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 21, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 21, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 22, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 22, "id_servico": 2, "quantidade": 1, "valor_unitario": 80.0},
  {"id_atendimento": 22, "id_servico": 6, "quantidade": 1, "valor_unitario": 99.0},
  {"id_atendimento": 23, "id_servico": 3, "quantidade": 1, "valor_unitario": 209.0},
  {"id_atendimento": 24, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 24, "id_servico": 11, "quantidade": 1, "valor_unitario": 650.0},
  {"id_atendimento": 25, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 26, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 26, "id_servico": 14, "quantidade": 2, "valor_unitario": 35.0},
  {"id_atendimento": 27, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 28, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 29, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 30, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 31, "id_servico": 3, "quantidade": 1, "valor_unitario": 209.0},
  {"id_atendimento": 31, "id_servico": 14, "quantidade": 1, "valor_unitario": 31.5},
  {"id_atendimento": 32, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 32, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 33, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 33, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 33, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 34, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 34, "id_servico": 8, "quantidade": 1, "valor_unitario": 230.0},
  {"id_atendimento": 34, "id_servico": 10, "quantidade": 1, "valor_unitario": 152.0},
  {"id_atendimento": 35, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 36, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 37, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 38, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 38, "id_servico": 13, "quantidade": 1, "valor_unitario": 90.0},
  {"id_atendimento": 39, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 40, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 40, "id_servico": 2, "quantidade": 1, "valor_unitario": 80.0},
  {"id_atendimento": 41, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 42, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 43, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 43, "id_servico": 2, "quantidade": 1, "valor_unitario": 76.0},
  {"id_atendimento": 44, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 44, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 45, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 45, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 45, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 46, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 46, "id_servico": 13, "quantidade": 1, "valor_unitario": 90.0},
  {"id_atendimento": 47, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 47, "id_servico": 2, "quantidade": 1, "valor_unitario": 80.0},
  {"id_atendimento": 47, "id_servico": 6, "quantidade": 1, "valor_unitario": 104.5},
  {"id_atendimento": 48, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 49, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 49, "id_servico": 2, "quantidade": 1, "valor_unitario": 80.0},
  {"id_atendimento": 50, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 51, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 51, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 52, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 53, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 54, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 55, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 55, "id_servico": 13, "quantidade": 1, "valor_unitario": 81.0},
  {"id_atendimento": 56, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 56, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 57, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 58, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 59, "id_servico": 3, "quantidade": 1, "valor_unitario": 209.0},
  {"id_atendimento": 59, "id_servico": 14, "quantidade": 1, "valor_unitario": 33.25},
  {"id_atendimento": 60, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 60, "id_servico": 2, "quantidade": 1, "valor_unitario": 76.0},
  {"id_atendimento": 61, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 62, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 63, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 64, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 65, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 66, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 66, "id_servico": 5, "quantidade": 1, "valor_unitario": 54.0},
  {"id_atendimento": 66, "id_servico": 13, "quantidade": 1, "valor_unitario": 90.0},
  {"id_atendimento": 67, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 67, "id_servico": 14, "quantidade": 2, "valor_unitario": 35.0},
  {"id_atendimento": 68, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 68, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 69, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 69, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 70, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 70, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 71, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 71, "id_servico": 14, "quantidade": 2, "valor_unitario": 31.5},
  {"id_atendimento": 72, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 73, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 73, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 74, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 74, "id_servico": 6, "quantidade": 1, "valor_unitario": 110.0},
  {"id_atendimento": 75, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 76, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 76, "id_servico": 9, "quantidade": 1, "valor_unitario": 180.0},
  {"id_atendimento": 76, "id_servico": 11, "quantidade": 1, "valor_unitario": 650.0},
  {"id_atendimento": 77, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 78, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 78, "id_servico": 14, "quantidade": 1, "valor_unitario": 31.5},
  {"id_atendimento": 79, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 79, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 79, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 80, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 81, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 81, "id_servico": 13, "quantidade": 1, "valor_unitario": 81.0},
  {"id_atendimento": 82, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 83, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 83, "id_servico": 14, "quantidade": 2, "valor_unitario": 35.0},
  {"id_atendimento": 84, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 85, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 85, "id_servico": 7, "quantidade": 1, "valor_unitario": 71.25},
  {"id_atendimento": 85, "id_servico": 14, "quantidade": 1, "valor_unitario": 33.25},
  {"id_atendimento": 86, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 86, "id_servico": 6, "quantidade": 1, "valor_unitario": 104.5},
  {"id_atendimento": 87, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 88, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 88, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 89, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 90, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 91, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 91, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 91, "id_servico": 14, "quantidade": 2, "valor_unitario": 35.0},
  {"id_atendimento": 92, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 93, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 93, "id_servico": 12, "quantidade": 1, "valor_unitario": 420.0},
  {"id_atendimento": 93, "id_servico": 14, "quantidade": 1, "valor_unitario": 33.25},
  {"id_atendimento": 94, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 94, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 95, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 96, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 96, "id_servico": 5, "quantidade": 1, "valor_unitario": 60.0},
  {"id_atendimento": 97, "id_servico": 3, "quantidade": 1, "valor_unitario": 209.0},
  {"id_atendimento": 98, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 98, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 99, "id_servico": 2, "quantidade": 1, "valor_unitario": 80.0},
  {"id_atendimento": 99, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 99, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 100, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 101, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 101, "id_servico": 12, "quantidade": 1, "valor_unitario": 420.0},
  {"id_atendimento": 102, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 102, "id_servico": 6, "quantidade": 1, "valor_unitario": 110.0},
  {"id_atendimento": 102, "id_servico": 13, "quantidade": 1, "valor_unitario": 90.0},
  {"id_atendimento": 103, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 103, "id_servico": 5, "quantidade": 1, "valor_unitario": 60.0},
  {"id_atendimento": 103, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 104, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 105, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 106, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 107, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 108, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 108, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 109, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 110, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 111, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 111, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 112, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 112, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 113, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 114, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 114, "id_servico": 13, "quantidade": 1, "valor_unitario": 90.0},
  {"id_atendimento": 115, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 115, "id_servico": 7, "quantidade": 1, "valor_unitario": 71.25},
  {"id_atendimento": 116, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 116, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 116, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 117, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 117, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 118, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 119, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 119, "id_servico": 13, "quantidade": 1, "valor_unitario": 85.5},
  {"id_atendimento": 120, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 121, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 121, "id_servico": 6, "quantidade": 1, "valor_unitario": 110.0},
  {"id_atendimento": 122, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 123, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 124, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 124, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 125, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 126, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 127, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 127, "id_servico": 6, "quantidade": 1, "valor_unitario": 110.0},
  {"id_atendimento": 127, "id_servico": 13, "quantidade": 1, "valor_unitario": 90.0},
  {"id_atendimento": 128, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 129, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 130, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 130, "id_servico": 14, "quantidade": 2, "valor_unitario": 35.0},
  {"id_atendimento": 131, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 131, "id_servico": 5, "quantidade": 1, "valor_unitario": 57.0},
  {"id_atendimento": 131, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 132, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 132, "id_servico": 7, "quantidade": 1, "valor_unitario": 71.25},
  {"id_atendimento": 133, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 133, "id_servico": 13, "quantidade": 1, "valor_unitario": 90.0},
  {"id_atendimento": 134, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 134, "id_servico": 7, "quantidade": 1, "valor_unitario": 67.5},
  {"id_atendimento": 135, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 136, "id_servico": 3, "quantidade": 1, "valor_unitario": 209.0},
  {"id_atendimento": 137, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 137, "id_servico": 7, "quantidade": 1, "valor_unitario": 71.25},
  {"id_atendimento": 137, "id_servico": 13, "quantidade": 1, "valor_unitario": 90.0},
  {"id_atendimento": 138, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 139, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 140, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 140, "id_servico": 7, "quantidade": 1, "valor_unitario": 67.5},
  {"id_atendimento": 141, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 141, "id_servico": 5, "quantidade": 1, "valor_unitario": 60.0},
  {"id_atendimento": 142, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 143, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 144, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 144, "id_servico": 4, "quantidade": 1, "valor_unitario": 95.0},
  {"id_atendimento": 144, "id_servico": 13, "quantidade": 1, "valor_unitario": 81.0},
  {"id_atendimento": 145, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 145, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 146, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 146, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 147, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 147, "id_servico": 7, "quantidade": 1, "valor_unitario": 67.5},
  {"id_atendimento": 147, "id_servico": 14, "quantidade": 1, "valor_unitario": 31.5},
  {"id_atendimento": 148, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 148, "id_servico": 2, "quantidade": 1, "valor_unitario": 72.0},
  {"id_atendimento": 149, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 150, "id_servico": 1, "quantidade": 1, "valor_unitario": 142.5},
  {"id_atendimento": 150, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 151, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 151, "id_servico": 14, "quantidade": 1, "valor_unitario": 35.0},
  {"id_atendimento": 152, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 152, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 153, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 154, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0},
  {"id_atendimento": 154, "id_servico": 12, "quantidade": 1, "valor_unitario": 420.0},
  {"id_atendimento": 154, "id_servico": 14, "quantidade": 1, "valor_unitario": 33.25},
  {"id_atendimento": 155, "id_servico": 3, "quantidade": 1, "valor_unitario": 198.0},
  {"id_atendimento": 156, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 156, "id_servico": 7, "quantidade": 1, "valor_unitario": 67.5},
  {"id_atendimento": 157, "id_servico": 3, "quantidade": 1, "valor_unitario": 209.0},
  {"id_atendimento": 157, "id_servico": 7, "quantidade": 1, "valor_unitario": 75.0},
  {"id_atendimento": 158, "id_servico": 1, "quantidade": 1, "valor_unitario": 150.0},
  {"id_atendimento": 158, "id_servico": 7, "quantidade": 1, "valor_unitario": 67.5},
  {"id_atendimento": 159, "id_servico": 1, "quantidade": 1, "valor_unitario": 135.0},
  {"id_atendimento": 160, "id_servico": 3, "quantidade": 1, "valor_unitario": 220.0}
]

Writing dados/atendimento_servico.json


### 7.2 Importação dos arquivos para o banco (respeitando a ordem das FKs: tabelas pai antes das filhas)

In [18]:
def carregar_csv(tabela):
    caminho = f'dados/{tabela}.csv'
    with open(caminho, encoding='utf-8') as f:
        colunas = f.readline().strip()
        f.seek(0)
        with conn.cursor() as cur:
            cur.copy_expert(f"COPY {tabela} ({colunas}) FROM STDIN WITH (FORMAT csv, HEADER true)", f)
    print(f'CSV  -> {tabela:<20} ok')

def carregar_json(tabela):
    registros = json.load(open(f'dados/{tabela}.json', encoding='utf-8'))
    colunas = list(registros[0].keys())
    sql = f"INSERT INTO {tabela} ({', '.join(colunas)}) VALUES ({', '.join(['%s'] * len(colunas))})"
    with conn.cursor() as cur:
        cur.executemany(sql, [tuple(r[c] for c in colunas) for r in registros])
    print(f'JSON -> {tabela:<20} {len(registros)} registros')

ordem = [('estado','csv'), ('cidade','csv'), ('tutor','csv'), ('especie','csv'), ('raca','csv'),
         ('animal','csv'), ('especialidade','csv'), ('veterinario','csv'), ('servico','json'),
         ('atendimento','csv'), ('atendimento_servico','json')]
for tabela, formato in ordem:
    carregar_csv(tabela) if formato == 'csv' else carregar_json(tabela)

# Como os IDs vieram dos arquivos, reposiciona as colunas IDENTITY para os próximos INSERTs
for tabela, formato in ordem:
    if tabela == 'atendimento_servico':
        continue
    pk = f'id_{tabela}'
    if formato == 'csv':
        maior = max(int(l[pk]) for l in csv.DictReader(open(f'dados/{tabela}.csv', encoding='utf-8')))
    else:
        maior = max(r[pk] for r in json.load(open(f'dados/{tabela}.json', encoding='utf-8')))
    executar(f'ALTER TABLE {tabela} ALTER COLUMN {pk} RESTART WITH {maior + 1}')
print('Carga concluída!')

CSV  -> estado               ok
CSV  -> cidade               ok
CSV  -> tutor                ok
CSV  -> especie              ok
CSV  -> raca                 ok
CSV  -> animal               ok
CSV  -> especialidade        ok
CSV  -> veterinario          ok
JSON -> servico              15 registros
CSV  -> atendimento          ok
JSON -> atendimento_servico  273 registros
Carga concluída!


Conferência da quantidade de registros carregados em cada tabela:

In [19]:
consulta("""
SELECT 'estado' AS tabela, COUNT(id_estado) AS registros FROM estado WHERE id_estado > 0
UNION ALL SELECT 'cidade', COUNT(id_cidade) FROM cidade WHERE id_cidade > 0
UNION ALL SELECT 'tutor', COUNT(id_tutor) FROM tutor WHERE id_tutor > 0
UNION ALL SELECT 'especie', COUNT(id_especie) FROM especie WHERE id_especie > 0
UNION ALL SELECT 'raca', COUNT(id_raca) FROM raca WHERE id_raca > 0
UNION ALL SELECT 'animal', COUNT(id_animal) FROM animal WHERE id_animal > 0
UNION ALL SELECT 'especialidade', COUNT(id_especialidade) FROM especialidade WHERE id_especialidade > 0
UNION ALL SELECT 'veterinario', COUNT(id_veterinario) FROM veterinario WHERE id_veterinario > 0
UNION ALL SELECT 'servico', COUNT(id_servico) FROM servico WHERE id_servico > 0
UNION ALL SELECT 'atendimento', COUNT(id_atendimento) FROM atendimento WHERE id_atendimento > 0
UNION ALL SELECT 'atendimento_servico', COUNT(id_servico) FROM atendimento_servico WHERE id_servico > 0;
""")

11 linha(s) retornada(s)


,tabela,registros
0,estado,5
1,cidade,12
2,tutor,40
3,especie,4
4,raca,14
5,animal,60
6,especialidade,6
7,veterinario,8
8,servico,15
9,atendimento,160


### 7.3 Script alternativo de carga com `INSERT`s
O mesmo conjunto de dados também foi gerado como script SQL de `INSERT`s (`inserts.sql`), respeitando a ordem das FKs. Ele serve como forma alternativa de carga em um banco recém-criado (rodar `schema.sql` e depois `inserts.sql`). Como os dados já foram carregados na seção 7.2, a célula abaixo **apenas grava o arquivo** — para usá-lo, basta descomentar a última célula desta seção.

In [20]:
%%writefile inserts.sql
-- Carga alternativa via INSERTs (Clínica Veterinária PetVida)

INSERT INTO estado (id_estado, sigla, nome) VALUES
  (1, 'SC', 'Santa Catarina'),
  (2, 'PR', 'Paraná'),
  (3, 'RS', 'Rio Grande do Sul'),
  (4, 'SP', 'São Paulo'),
  (5, 'RJ', 'Rio de Janeiro');

INSERT INTO cidade (id_cidade, nome, id_estado) VALUES
  (1, 'Blumenau', 1),
  (2, 'Florianópolis', 1),
  (3, 'Joinville', 1),
  (4, 'Itajaí', 1),
  (5, 'Curitiba', 2),
  (6, 'Londrina', 2),
  (7, 'Porto Alegre', 3),
  (8, 'Caxias do Sul', 3),
  (9, 'São Paulo', 4),
  (10, 'Campinas', 4),
  (11, 'Rio de Janeiro', 5),
  (12, 'Niterói', 5);

INSERT INTO tutor (id_tutor, nome, cpf, email, telefone, data_cadastro, id_cidade) VALUES
  (1, 'Ana Santos', '242.854.204-11', 'ana.santos1@gmail.com', '(21) 95506-5012', '2024-08-16', 11),
  (2, 'Bruno Schmidt', '195.323.338-12', 'bruno.schmidt2@gmail.com', '(11) 97912-1520', '2024-01-31', 9),
  (3, 'Carla Barbosa', '325.559.703-13', 'carla.barbosa3@gmail.com', '(11) 94257-9928', '2025-03-05', 5),
  (4, 'Daniel Silva', '259.320.881-14', 'daniel.silva4@hotmail.com', '(21) 97924-6574', '2024-10-11', 6),
  (5, 'Eduarda Santos', '452.718.370-15', 'eduarda.santos5@gmail.com', '(51) 92584-6881', '2026-05-17', 1),
  (6, 'Felipe Martins', '400.949.743-16', 'felipe.martins6@gmail.com', '(47) 97201-2291', '2025-07-19', 10),
  (7, 'Gabriela Ferreira', '333.891.396-17', 'gabriela.ferreira7@gmail.com', '(48) 92139-1750', '2025-11-08', 2),
  (8, 'Henrique Almeida', '954.473.266-18', 'henrique.almeida8@gmail.com', '(51) 95554-8428', '2025-10-12', 6),
  (9, 'Isabela Ferreira', '750.275.646-19', 'isabela.ferreira9@hotmail.com', '(21) 95374-2169', '2025-09-15', 12),
  (10, 'João Almeida', '804.670.324-20', 'joao.almeida10@hotmail.com', '(51) 97216-5422', '2025-10-17', 11),
  (11, 'Larissa Araújo', '374.167.316-21', 'larissa.araujo11@gmail.com', '(48) 91525-6168', '2025-02-14', 10),
  (12, 'Marcos Araújo', '569.246.371-22', 'marcos.araujo12@hotmail.com', '(21) 99179-7482', '2025-10-20', 3),
  (13, 'Mariana Almeida', '508.470.324-23', 'mariana.almeida13@gmail.com', '(11) 95304-8019', '2025-08-20', 3),
  (14, 'Natália Ribeiro', '742.263.911-24', 'natalia.ribeiro14@yahoo.com.br', '(47) 91771-2796', '2024-06-05', 11),
  (15, 'Otávio Gomes', '579.641.357-25', 'otavio.gomes15@gmail.com', '(47) 97304-7252', '2025-09-02', 9),
  (16, 'Paula Silva', '756.448.214-26', 'paula.silva16@gmail.com', '(21) 99797-5371', '2026-02-26', 5),
  (17, 'Rafael Gomes', '880.282.619-27', 'rafael.gomes17@hotmail.com', '(51) 91053-5315', '2025-05-27', 2),
  (18, 'Renata Lima', '880.265.652-28', 'renata.lima18@gmail.com', '(11) 94258-3504', '2025-01-17', 9),
  (19, 'Sérgio Silva', '471.999.951-29', 'sergio.silva19@gmail.com', '(41) 99005-1319', '2024-04-24', 5),
  (20, 'Tatiane Almeida', '597.935.170-30', 'tatiane.almeida20@gmail.com', '(48) 92290-2403', '2026-01-19', 9),
  (21, 'Lucas Pereira', '640.993.721-31', 'lucas.pereira21@hotmail.com', '(21) 98787-3705', '2024-09-28', 7),
  (22, 'Juliana Rodrigues', '787.765.482-32', 'juliana.rodrigues22@gmail.com', '(21) 94295-6107', '2025-02-12', 8),
  (23, 'Marcelo Ribeiro', '446.121.702-33', 'marcelo.ribeiro23@yahoo.com.br', '(47) 95061-4681', '2024-03-06', 9),
  (24, 'Fernanda Almeida', '746.160.334-34', 'fernanda.almeida24@gmail.com', '(48) 91117-2163', '2025-12-25', 2),
  (25, 'Gustavo Souza', '785.597.319-35', 'gustavo.souza25@outlook.com', '(47) 99423-4899', '2024-10-12', 9),
  (26, 'Camila Pereira', '584.926.516-36', 'camila.pereira26@gmail.com', '(11) 98744-4981', '2026-03-14', 4),
  (27, 'Rodrigo Santos', '520.578.984-37', 'rodrigo.santos27@gmail.com', '(21) 98062-6804', '2025-03-09', 12),
  (28, 'Patrícia Souza', '982.211.354-38', 'patricia.souza28@gmail.com', '(47) 97596-6559', '2026-03-30', 4),
  (29, 'Thiago Rodrigues', '385.573.355-39', 'thiago.rodrigues29@gmail.com', '(51) 93296-7912', '2024-07-06', 2),
  (30, 'Beatriz Martins', '115.195.871-40', 'beatriz.martins30@gmail.com', '(47) 91828-9856', '2026-05-06', 4),
  (31, 'Leonardo Costa', '510.160.268-41', 'leonardo.costa31@yahoo.com.br', '(51) 98886-4502', '2026-06-04', 7),
  (32, 'Amanda Silva', '813.848.902-42', 'amanda.silva32@yahoo.com.br', '(41) 98454-5673', '2025-03-09', 9),
  (33, 'Vinícius Rocha', '693.853.655-43', 'vinicius.rocha33@hotmail.com', '(48) 95861-4566', '2024-02-29', 1),
  (34, 'Letícia Araújo', '643.261.158-44', 'leticia.araujo34@gmail.com', '(47) 98811-9238', '2026-05-23', 9),
  (35, 'André Oliveira', '222.683.352-45', 'andre.oliveira35@hotmail.com', '(47) 92113-4853', '2025-02-17', 10),
  (36, 'Aline Barbosa', '697.678.635-46', 'aline.barbosa36@gmail.com', '(11) 92343-7868', '2025-11-04', 6),
  (37, 'Diego Nascimento', '505.234.787-47', 'diego.nascimento37@hotmail.com', '(21) 96147-4910', '2024-09-28', 11),
  (38, 'Bianca Lima', '736.676.202-48', 'bianca.lima38@yahoo.com.br', '(41) 92188-1152', '2025-04-14', 2),
  (39, 'Matheus Schmidt', '170.350.478-49', 'matheus.schmidt39@hotmail.com', '(11) 95345-3170', '2024-12-23', 5),
  (40, 'Priscila Costa', '783.936.667-50', 'priscila.costa40@yahoo.com.br', '(11) 95956-9666', '2024-01-09', 5);

INSERT INTO especie (id_especie, nome) VALUES
  (1, 'Cão'),
  (2, 'Gato'),
  (3, 'Ave'),
  (4, 'Roedor');

INSERT INTO raca (id_raca, nome, id_especie) VALUES
  (1, 'Labrador Retriever', 1),
  (2, 'Golden Retriever', 1),
  (3, 'Shih Tzu', 1),
  (4, 'Bulldog Francês', 1),
  (5, 'Pastor Alemão', 1),
  (6, 'SRD (Cão)', 1),
  (7, 'Siamês', 2),
  (8, 'Persa', 2),
  (9, 'Maine Coon', 2),
  (10, 'SRD (Gato)', 2),
  (11, 'Calopsita', 3),
  (12, 'Periquito Australiano', 3),
  (13, 'Hamster Sírio', 4),
  (14, 'Porquinho-da-índia', 4);

INSERT INTO animal (id_animal, nome, data_nascimento, sexo, peso_kg, id_tutor, id_raca) VALUES
  (1, 'Thor', '2014-04-28', 'M', 0.04, 1, 12),
  (2, 'Mel', '2024-05-29', 'M', 0.09, 2, 11),
  (3, 'Luna', '2019-09-09', 'M', 5.06, 3, 7),
  (4, 'Bob', '2023-05-03', 'F', 4.0, 4, 10),
  (5, 'Nina', '2013-02-20', 'M', 0.11, 5, 11),
  (6, 'Pipoca', '2012-12-27', 'M', 4.33, 6, 10),
  (7, 'Max', '2017-11-15', 'M', 34.64, 7, 2),
  (8, 'Bela', '2021-08-04', 'M', 7.8, 8, 6),
  (9, 'Fred', '2015-05-06', 'M', 0.18, 9, 13),
  (10, 'Lola', '2015-04-28', 'F', 8.19, 10, 6),
  (11, 'Toby', '2012-11-22', 'F', 5.05, 11, 3),
  (12, 'Amora', '2014-04-22', 'F', 7.9, 12, 3),
  (13, 'Simba', '2021-02-11', 'M', 0.12, 13, 11),
  (14, 'Frida', '2015-08-24', 'M', 8.41, 14, 3),
  (15, 'Zeus', '2019-06-14', 'F', 37.03, 15, 1),
  (16, 'Pandora', '2017-07-25', 'F', 3.64, 16, 10),
  (17, 'Paçoca', '2020-07-30', 'M', 5.58, 17, 8),
  (18, 'Kiara', '2022-04-28', 'F', 5.53, 18, 3),
  (19, 'Bidu', '2017-02-07', 'M', 3.09, 19, 10),
  (20, 'Cacau', '2019-05-13', 'F', 36.24, 20, 2),
  (21, 'Romeu', '2018-04-05', 'F', 0.16, 21, 13),
  (22, 'Julieta', '2024-01-10', 'F', 43.97, 22, 5),
  (23, 'Fumaça', '2017-11-09', 'M', 32.55, 23, 2),
  (24, 'Mia', '2012-11-09', 'M', 1.0, 24, 14),
  (25, 'Rex', '2019-01-13', 'F', 11.64, 25, 4),
  (26, 'Jade', '2020-08-21', 'M', 11.37, 26, 6),
  (27, 'Chico', '2012-01-14', 'M', 4.09, 27, 8),
  (28, 'Belinha', '2019-05-28', 'F', 34.95, 28, 1),
  (29, 'Ozzy', '2018-09-26', 'F', 33.67, 29, 2),
  (30, 'Maya', '2018-08-19', 'M', 9.15, 30, 4),
  (31, 'Pingo', '2020-07-02', 'M', 4.85, 31, 8),
  (32, 'Sofia', '2024-04-15', 'M', 5.52, 32, 3),
  (33, 'Duke', '2025-01-03', 'F', 6.32, 33, 3),
  (34, 'Estrela', '2016-10-16', 'F', 41.49, 34, 5),
  (35, 'Tobias', '2015-10-22', 'M', 0.13, 35, 13),
  (36, 'Lua', '2019-07-08', 'M', 5.46, 36, 8),
  (37, 'Nala', '2018-12-17', 'M', 9.03, 37, 9),
  (38, 'Pretinha', '2013-01-13', 'M', 37.79, 38, 2),
  (39, 'Floquinho', '2013-08-19', 'F', 4.24, 39, 7),
  (40, 'Bolinha', '2016-05-11', 'F', 4.48, 40, 7),
  (41, 'Margarida', '2012-02-15', 'M', 7.89, 15, 3),
  (42, 'Apolo', '2023-08-14', 'F', 25.65, 16, 2),
  (43, 'Safira', '2014-09-21', 'F', 0.03, 30, 12),
  (44, 'Sansão', '2024-07-14', 'F', 5.85, 29, 8),
  (45, 'Lili', '2023-04-28', 'F', 5.49, 36, 7),
  (46, 'Bento', '2015-07-27', 'F', 16.25, 16, 6),
  (47, 'Pérola', '2018-03-21', 'F', 5.51, 18, 10),
  (48, 'Marley', '2018-05-29', 'M', 32.62, 21, 5),
  (49, 'Tieta', '2013-10-22', 'M', 0.09, 25, 11),
  (50, 'Kiko', '2016-10-18', 'M', 4.24, 22, 8),
  (51, 'Dora', '2021-04-29', 'M', 10.17, 27, 6),
  (52, 'Hulk', '2025-02-05', 'M', 42.56, 37, 5),
  (53, 'Pituca', '2012-02-18', 'F', 9.79, 25, 4),
  (54, 'Billy', '2021-05-25', 'M', 4.95, 18, 10),
  (55, 'Princesa', '2012-08-25', 'F', 33.71, 26, 5),
  (56, 'Teodoro', '2022-06-25', 'M', 9.9, 35, 9),
  (57, 'Laila', '2020-11-01', 'M', 26.26, 28, 1),
  (58, 'Gaia', '2022-05-10', 'M', 25.65, 25, 2),
  (59, 'Brisa', '2022-03-13', 'F', 10.03, 25, 4),
  (60, 'Faísca', '2021-06-15', 'F', 8.17, 31, 3);

INSERT INTO especialidade (id_especialidade, nome) VALUES
  (1, 'Clínica Geral'),
  (2, 'Dermatologia'),
  (3, 'Cardiologia'),
  (4, 'Ortopedia'),
  (5, 'Animais Silvestres e Exóticos'),
  (6, 'Odontologia');

INSERT INTO veterinario (id_veterinario, nome, crmv, id_especialidade, data_admissao) VALUES
  (1, 'Dra. Helena Duarte', 'CRMV-SC 10234', 1, '2018-03-01'),
  (2, 'Dr. Ricardo Mendes', 'CRMV-SC 11876', 4, '2019-07-15'),
  (3, 'Dra. Sabrina Kowalski', 'CRMV-PR 08451', 2, '2020-02-10'),
  (4, 'Dr. Fábio Teixeira', 'CRMV-SC 12990', 3, '2021-05-03'),
  (5, 'Dra. Lívia Moraes', 'CRMV-SC 13402', 5, '2022-01-20'),
  (6, 'Dr. Caio Batista', 'CRMV-RS 07765', 6, '2022-09-12'),
  (7, 'Dra. Viviane Lopes', 'CRMV-SC 14051', 1, '2023-04-01'),
  (8, 'Dr. Eduardo Prado', 'CRMV-SP 21456', 1, '2024-08-19');

INSERT INTO servico (id_servico, descricao, categoria, preco_base, ativo) VALUES
  (1, 'Consulta clínica', 'Consulta', 150.0, TRUE),
  (2, 'Consulta de retorno', 'Consulta', 80.0, TRUE),
  (3, 'Consulta especializada', 'Consulta', 220.0, TRUE),
  (4, 'Vacina V10 (cães)', 'Vacina', 95.0, TRUE),
  (5, 'Vacina antirrábica', 'Vacina', 60.0, TRUE),
  (6, 'Vacina V4 felina', 'Vacina', 110.0, TRUE),
  (7, 'Hemograma completo', 'Exame', 75.0, TRUE),
  (8, 'Ultrassonografia abdominal', 'Exame', 230.0, TRUE),
  (9, 'Raio-X', 'Exame', 180.0, TRUE),
  (10, 'Eletrocardiograma', 'Exame', 160.0, TRUE),
  (11, 'Castração', 'Cirurgia', 650.0, TRUE),
  (12, 'Limpeza de tártaro', 'Cirurgia', 420.0, TRUE),
  (13, 'Banho e tosa', 'Estética', 90.0, TRUE),
  (14, 'Aplicação de medicamento', 'Procedimento', 35.0, TRUE),
  (15, 'Microchipagem', 'Procedimento', 120.0, FALSE);

INSERT INTO atendimento (id_atendimento, data_hora, id_animal, id_veterinario, status, observacao) VALUES
  (1, '2026-01-09 15:30:00', 44, 6, 'Concluído', 'Prescrito antibiótico'),
  (2, '2026-01-14 16:00:00', 23, 1, 'Concluído', 'Acompanhamento semestral'),
  (3, '2026-01-20 08:30:00', 55, 3, 'Concluído', 'Sem alterações'),
  (4, '2026-01-23 12:00:00', 3, 7, 'Cancelado', 'Cancelado pelo tutor'),
  (5, '2026-01-26 12:30:00', 3, 6, 'Concluído', 'Animal estável'),
  (6, '2026-01-26 13:30:00', 18, 7, 'Concluído', 'Animal estável'),
  (7, '2026-01-30 09:30:00', 29, 7, 'Concluído', 'Sem alterações'),
  (8, '2026-02-02 09:00:00', 20, 1, 'Concluído', 'Acompanhamento semestral'),
  (9, '2026-02-04 09:00:00', 35, 5, 'Concluído', 'Animal estável'),
  (10, '2026-02-05 15:00:00', 52, 8, 'Concluído', 'Acompanhamento semestral'),
  (11, '2026-02-09 11:30:00', 24, 5, 'Concluído', 'Dieta recomendada'),
  (12, '2026-02-09 15:30:00', 41, 2, 'Concluído', 'Animal estável'),
  (13, '2026-02-10 14:00:00', 28, 1, 'Cancelado', 'Cancelado pelo tutor'),
  (14, '2026-02-11 17:00:00', 10, 1, 'Concluído', 'Prescrito antibiótico'),
  (15, '2026-02-12 10:00:00', 56, 7, 'Concluído', 'Sem alterações'),
  (16, '2026-02-14 08:30:00', 14, 8, 'Concluído', 'Acompanhamento semestral'),
  (17, '2026-02-14 13:30:00', 42, 1, 'Concluído', 'Retorno em 30 dias'),
  (18, '2026-02-18 12:30:00', 16, 7, 'Concluído', 'Acompanhamento semestral'),
  (19, '2026-02-19 14:00:00', 16, 4, 'Concluído', 'Animal estável'),
  (20, '2026-02-23 10:00:00', 31, 2, 'Concluído', 'Prescrito antibiótico'),
  (21, '2026-02-23 13:30:00', 21, 5, 'Concluído', 'Prescrito antibiótico'),
  (22, '2026-02-23 16:30:00', 19, 1, 'Concluído', 'Dieta recomendada'),
  (23, '2026-02-25 14:30:00', 46, 4, 'Concluído', 'Retorno em 30 dias'),
  (24, '2026-02-25 15:00:00', 45, 2, 'Concluído', 'Prescrito antibiótico'),
  (25, '2026-02-25 16:00:00', 43, 5, 'Concluído', 'Dieta recomendada'),
  (26, '2026-02-27 10:30:00', 27, 7, 'Concluído', 'Sem alterações'),
  (27, '2026-02-28 14:30:00', 41, 8, 'Concluído', 'Retorno em 30 dias'),
  (28, '2026-03-02 14:30:00', 51, 8, 'Concluído', 'Acompanhamento semestral'),
  (29, '2026-03-04 16:00:00', 42, 1, 'Concluído', 'Retorno em 30 dias'),
  (30, '2026-03-06 16:30:00', 39, 6, 'Concluído', 'Dieta recomendada'),
  (31, '2026-03-07 08:30:00', 48, 3, 'Concluído', 'Dieta recomendada'),
  (32, '2026-03-09 11:00:00', 16, 7, 'Concluído', 'Dieta recomendada'),
  (33, '2026-03-09 12:00:00', 49, 5, 'Concluído', 'Sem alterações'),
  (34, '2026-03-10 13:00:00', 29, 4, 'Concluído', 'Retorno em 30 dias'),
  (35, '2026-03-12 13:00:00', 21, 5, 'Concluído', 'Animal estável'),
  (36, '2026-03-18 09:00:00', 19, 8, 'Concluído', 'Sem alterações'),
  (37, '2026-03-20 10:30:00', 52, 1, 'Concluído', 'Animal estável'),
  (38, '2026-03-21 11:00:00', 10, 1, 'Concluído', 'Retorno em 30 dias'),
  (39, '2026-03-23 15:00:00', 9, 5, 'Concluído', 'Dieta recomendada'),
  (40, '2026-03-24 10:00:00', 54, 7, 'Concluído', 'Retorno em 30 dias'),
  (41, '2026-03-26 08:30:00', 29, 7, 'Concluído', 'Animal estável'),
  (42, '2026-03-26 14:30:00', 31, 7, 'Concluído', 'Prescrito antibiótico'),
  (43, '2026-03-28 11:00:00', 56, 1, 'Concluído', 'Prescrito antibiótico'),
  (44, '2026-04-03 08:00:00', 46, 4, 'Concluído', 'Dieta recomendada'),
  (45, '2026-04-07 10:30:00', 5, 5, 'Concluído', 'Acompanhamento semestral'),
  (46, '2026-04-11 11:30:00', 15, 8, 'Concluído', 'Prescrito antibiótico'),
  (47, '2026-04-13 09:30:00', 6, 7, 'Concluído', 'Dieta recomendada'),
  (48, '2026-04-13 12:30:00', 56, 7, 'Concluído', 'Prescrito antibiótico'),
  (49, '2026-04-16 09:00:00', 7, 7, 'Concluído', 'Acompanhamento semestral'),
  (50, '2026-04-16 14:30:00', 7, 1, 'Concluído', 'Dieta recomendada'),
  (51, '2026-04-20 09:00:00', 38, 1, 'Concluído', 'Prescrito antibiótico'),
  (52, '2026-04-20 14:30:00', 56, 8, 'Concluído', 'Acompanhamento semestral'),
  (53, '2026-04-21 15:30:00', 31, 3, 'Cancelado', 'Cancelado pelo tutor'),
  (54, '2026-04-24 13:30:00', 13, 5, 'Concluído', 'Retorno em 30 dias'),
  (55, '2026-04-27 11:30:00', 44, 7, 'Concluído', 'Acompanhamento semestral'),
  (56, '2026-04-27 14:00:00', 1, 5, 'Concluído', 'Retorno em 30 dias'),
  (57, '2026-04-28 09:30:00', 59, 7, 'Concluído', 'Animal estável'),
  (58, '2026-04-28 17:00:00', 47, 7, 'Concluído', 'Dieta recomendada'),
  (59, '2026-04-29 09:30:00', 55, 6, 'Concluído', 'Prescrito antibiótico'),
  (60, '2026-05-06 12:00:00', 4, 1, 'Concluído', 'Dieta recomendada'),
  (61, '2026-05-06 15:30:00', 49, 5, 'Concluído', 'Sem alterações'),
  (62, '2026-05-06 17:30:00', 2, 5, 'Concluído', 'Animal estável'),
  (63, '2026-05-08 15:30:00', 29, 7, 'Concluído', 'Retorno em 30 dias'),
  (64, '2026-05-09 08:00:00', 13, 5, 'Concluído', 'Prescrito antibiótico'),
  (65, '2026-05-11 11:00:00', 40, 1, 'Concluído', 'Acompanhamento semestral'),
  (66, '2026-05-13 11:00:00', 47, 1, 'Concluído', 'Dieta recomendada'),
  (67, '2026-05-14 13:00:00', 50, 3, 'Concluído', 'Animal estável'),
  (68, '2026-05-20 17:00:00', 47, 1, 'Concluído', 'Dieta recomendada'),
  (69, '2026-05-20 17:30:00', 13, 5, 'Cancelado', 'Cancelado pelo tutor'),
  (70, '2026-05-21 09:30:00', 6, 3, 'Cancelado', 'Cancelado pelo tutor'),
  (71, '2026-05-25 10:00:00', 40, 7, 'Concluído', 'Animal estável'),
  (72, '2026-05-26 16:00:00', 34, 8, 'Concluído', 'Acompanhamento semestral'),
  (73, '2026-05-27 11:00:00', 2, 5, 'Concluído', 'Sem alterações'),
  (74, '2026-05-29 15:00:00', 44, 1, 'Concluído', 'Acompanhamento semestral'),
  (75, '2026-05-30 09:30:00', 13, 5, 'Concluído', 'Dieta recomendada'),
  (76, '2026-05-30 15:30:00', 29, 2, 'Cancelado', 'Cancelado pelo tutor'),
  (77, '2026-05-30 15:30:00', 57, 8, 'Concluído', 'Sem alterações'),
  (78, '2026-06-02 09:30:00', 2, 5, 'Cancelado', 'Cancelado pelo tutor'),
  (79, '2026-06-05 08:30:00', 18, 1, 'Concluído', 'Acompanhamento semestral'),
  (80, '2026-06-06 17:30:00', 20, 6, 'Concluído', 'Acompanhamento semestral'),
  (81, '2026-06-09 15:00:00', 7, 1, 'Concluído', 'Sem alterações'),
  (82, '2026-06-09 17:00:00', 33, 7, 'Concluído', 'Dieta recomendada'),
  (83, '2026-06-10 17:30:00', 31, 6, 'Concluído', 'Acompanhamento semestral'),
  (84, '2026-06-11 17:30:00', 29, 1, 'Concluído', 'Acompanhamento semestral'),
  (85, '2026-06-12 15:00:00', 24, 5, 'Concluído', 'Dieta recomendada'),
  (86, '2026-06-12 15:30:00', 3, 1, 'Concluído', 'Animal estável'),
  (87, '2026-06-13 09:00:00', 60, 2, 'Concluído', 'Sem alterações'),
  (88, '2026-06-18 17:00:00', 5, 5, 'Concluído', 'Dieta recomendada'),
  (89, '2026-06-22 10:00:00', 42, 7, 'Concluído', 'Animal estável'),
  (90, '2026-06-22 12:30:00', 29, 6, 'Concluído', 'Acompanhamento semestral'),
  (91, '2026-06-22 17:30:00', 13, 5, 'Concluído', 'Sem alterações'),
  (92, '2026-06-23 10:30:00', 42, 1, 'Concluído', 'Acompanhamento semestral'),
  (93, '2026-06-24 15:30:00', 10, 6, 'Concluído', 'Prescrito antibiótico'),
  (94, '2026-06-25 10:00:00', 3, 2, 'Concluído', 'Acompanhamento semestral'),
  (95, '2026-06-27 10:30:00', 45, 4, 'Concluído', 'Retorno em 30 dias'),
  (96, '2026-06-27 11:30:00', 50, 1, 'Concluído', 'Animal estável'),
  (97, '2026-06-30 11:00:00', 50, 6, 'Concluído', 'Retorno em 30 dias'),
  (98, '2026-07-01 16:30:00', 43, 5, 'Concluído', 'Dieta recomendada'),
  (99, '2026-07-02 08:30:00', 4, 3, 'Concluído', 'Sem alterações'),
  (100, '2026-07-03 12:30:00', 9, 5, 'Concluído', 'Animal estável'),
  (101, '2026-07-06 14:00:00', 19, 6, 'Concluído', 'Dieta recomendada'),
  (102, '2026-07-11 16:30:00', 54, 1, 'Cancelado', 'Cancelado pelo tutor'),
  (103, '2026-07-13 11:30:00', 29, 8, 'Concluído', 'Animal estável'),
  (104, '2026-07-13 12:30:00', 50, 8, 'Concluído', 'Animal estável'),
  (105, '2026-07-14 09:00:00', 31, 1, 'Cancelado', 'Cancelado pelo tutor'),
  (106, '2026-07-22 14:30:00', 44, 6, 'Concluído', 'Acompanhamento semestral'),
  (107, '2026-07-25 16:30:00', 16, 1, 'Concluído', 'Animal estável'),
  (108, '2026-07-27 09:30:00', 55, 7, 'Concluído', 'Prescrito antibiótico'),
  (109, '2026-07-29 08:00:00', 24, 5, 'Concluído', 'Animal estável'),
  (110, '2026-07-31 12:30:00', 5, 5, 'Concluído', 'Retorno em 30 dias'),
  (111, '2026-07-31 13:00:00', 2, 5, 'Concluído', 'Dieta recomendada'),
  (112, '2026-08-06 10:00:00', 34, 8, 'Concluído', 'Dieta recomendada'),
  (113, '2026-08-13 10:00:00', 34, 3, 'Concluído', 'Retorno em 30 dias'),
  (114, '2026-08-14 09:30:00', 52, 1, 'Concluído', 'Dieta recomendada'),
  (115, '2026-08-17 14:30:00', 58, 8, 'Concluído', 'Retorno em 30 dias'),
  (116, '2026-08-18 09:30:00', 1, 5, 'Concluído', 'Sem alterações'),
  (117, '2026-08-19 12:00:00', 38, 4, 'Concluído', 'Animal estável'),
  (118, '2026-08-21 11:30:00', 45, 8, 'Concluído', 'Animal estável'),
  (119, '2026-08-24 10:30:00', 6, 7, 'Cancelado', 'Cancelado pelo tutor'),
  (120, '2026-08-24 13:30:00', 29, 7, 'Concluído', 'Dieta recomendada'),
  (121, '2026-08-31 11:30:00', 39, 1, 'Concluído', 'Prescrito antibiótico'),
  (122, '2026-08-31 12:30:00', 11, 6, 'Cancelado', 'Cancelado pelo tutor'),
  (123, '2026-09-02 08:30:00', 24, 5, 'Concluído', 'Animal estável'),
  (124, '2026-09-02 12:00:00', 55, 1, 'Concluído', 'Prescrito antibiótico'),
  (125, '2026-09-09 12:30:00', 7, 8, 'Concluído', 'Prescrito antibiótico'),
  (126, '2026-09-10 09:00:00', 41, 7, 'Concluído', 'Retorno em 30 dias'),
  (127, '2026-09-14 15:30:00', 44, 8, 'Concluído', 'Acompanhamento semestral'),
  (128, '2026-09-15 10:30:00', 57, 1, 'Concluído', 'Dieta recomendada'),
  (129, '2026-09-15 13:00:00', 32, 1, 'Concluído', 'Prescrito antibiótico'),
  (130, '2026-09-18 16:30:00', 29, 1, 'Concluído', 'Sem alterações'),
  (131, '2026-09-22 10:00:00', 18, 7, 'Concluído', 'Prescrito antibiótico'),
  (132, '2026-09-22 12:30:00', 54, 4, 'Concluído', 'Sem alterações'),
  (133, '2026-09-24 09:30:00', 28, 1, 'Concluído', 'Dieta recomendada'),
  (134, '2026-09-28 13:30:00', 41, 4, 'Concluído', 'Prescrito antibiótico'),
  (135, '2026-09-29 12:30:00', 30, 4, 'Concluído', 'Prescrito antibiótico'),
  (136, '2026-10-01 09:00:00', 31, 4, 'Concluído', 'Dieta recomendada'),
  (137, '2026-10-05 10:30:00', 3, 8, 'Concluído', 'Acompanhamento semestral'),
  (138, '2026-10-06 08:00:00', 46, 1, 'Concluído', 'Dieta recomendada'),
  (139, '2026-10-06 09:30:00', 24, 5, 'Concluído', 'Acompanhamento semestral'),
  (140, '2026-10-08 14:00:00', 57, 3, 'Concluído', 'Sem alterações'),
  (141, '2026-10-09 12:30:00', 55, 1, 'Concluído', 'Acompanhamento semestral'),
  (142, '2026-10-13 12:00:00', 9, 5, 'Cancelado', 'Cancelado pelo tutor'),
  (143, '2026-10-14 11:30:00', 29, 8, 'Concluído', 'Sem alterações'),
  (144, '2026-10-14 14:00:00', 15, 8, 'Cancelado', 'Cancelado pelo tutor'),
  (145, '2026-10-17 08:30:00', 2, 5, 'Concluído', 'Acompanhamento semestral'),
  (146, '2026-10-17 13:00:00', 13, 5, 'Concluído', 'Sem alterações'),
  (147, '2026-10-19 13:30:00', 34, 3, 'Cancelado', 'Cancelado pelo tutor'),
  (148, '2026-10-22 11:00:00', 40, 8, 'Cancelado', 'Cancelado pelo tutor'),
  (149, '2026-10-26 16:30:00', 58, 8, 'Concluído', 'Retorno em 30 dias'),
  (150, '2026-10-26 17:00:00', 49, 5, 'Concluído', 'Prescrito antibiótico'),
  (151, '2026-10-27 12:00:00', 50, 6, 'Concluído', 'Animal estável'),
  (152, '2026-10-28 15:00:00', 29, 5, 'Concluído', 'Dieta recomendada'),
  (153, '2026-11-04 13:30:00', 46, 6, 'Concluído', 'Animal estável'),
  (154, '2026-11-05 16:30:00', 30, 6, 'Cancelado', 'Cancelado pelo tutor'),
  (155, '2026-11-06 16:30:00', 30, 4, 'Concluído', 'Acompanhamento semestral'),
  (156, '2026-11-13 14:00:00', 32, 8, 'Concluído', 'Sem alterações'),
  (157, '2026-11-14 09:00:00', 20, 3, 'Concluído', 'Sem alterações'),
  (158, '2026-11-17 09:00:00', 41, 7, 'Concluído', 'Retorno em 30 dias'),
  (159, '2026-11-20 17:30:00', 54, 8, 'Concluído', 'Animal estável'),
  (160, '2026-11-21 11:30:00', 43, 5, 'Concluído', 'Retorno em 30 dias');

INSERT INTO atendimento_servico (id_atendimento, id_servico, quantidade, valor_unitario) VALUES
  (1, 3, 1, 198.0),
  (1, 12, 1, 420.0),
  (1, 14, 1, 35.0),
  (2, 1, 1, 150.0),
  (3, 3, 1, 220.0),
  (3, 7, 1, 75.0),
  (4, 1, 1, 142.5),
  (4, 7, 1, 67.5),
  (4, 14, 2, 35.0),
  (5, 3, 1, 220.0),
  (5, 12, 1, 399.0),
  (5, 14, 2, 35.0),
  (6, 1, 1, 135.0),
  (7, 1, 1, 150.0),
  (8, 1, 1, 142.5),
  (9, 1, 1, 150.0),
  (9, 3, 1, 220.0),
  (9, 7, 1, 75.0),
  (10, 1, 1, 135.0),
  (11, 3, 1, 220.0),
  (12, 3, 1, 220.0),
  (13, 1, 1, 150.0),
  (13, 14, 1, 31.5),
  (14, 1, 1, 150.0),
  (15, 1, 1, 150.0),
  (15, 6, 1, 110.0),
  (15, 14, 2, 35.0),
  (16, 1, 1, 150.0),
  (16, 7, 1, 75.0),
  (17, 1, 1, 142.5),
  (17, 5, 1, 54.0),
  (18, 1, 1, 150.0),
  (19, 3, 1, 220.0),
  (20, 3, 1, 220.0),
  (21, 3, 1, 220.0),
  (21, 7, 1, 75.0),
  (21, 14, 1, 35.0),
  (22, 1, 1, 142.5),
  (22, 2, 1, 80.0),
  (22, 6, 1, 99.0),
  (23, 3, 1, 209.0),
  (24, 3, 1, 220.0),
  (24, 11, 1, 650.0),
  (25, 3, 1, 220.0),
  (26, 1, 1, 150.0),
  (26, 14, 2, 35.0),
  (27, 1, 1, 135.0),
  (28, 1, 1, 150.0),
  (29, 1, 1, 150.0),
  (30, 3, 1, 220.0),
  (31, 3, 1, 209.0),
  (31, 14, 1, 31.5),
  (32, 1, 1, 142.5),
  (32, 7, 1, 75.0),
  (33, 1, 1, 150.0),
  (33, 3, 1, 220.0),
  (33, 7, 1, 75.0),
  (34, 3, 1, 220.0),
  (34, 8, 1, 230.0),
  (34, 10, 1, 152.0),
  (35, 3, 1, 220.0),
  (36, 1, 1, 150.0),
  (37, 1, 1, 150.0),
  (38, 1, 1, 150.0),
  (38, 13, 1, 90.0),
  (39, 3, 1, 220.0),
  (40, 1, 1, 150.0),
  (40, 2, 1, 80.0),
  (41, 1, 1, 135.0),
  (42, 1, 1, 135.0),
  (43, 1, 1, 135.0),
  (43, 2, 1, 76.0),
  (44, 3, 1, 220.0),
  (44, 7, 1, 75.0),
  (45, 1, 1, 150.0),
  (45, 3, 1, 220.0),
  (45, 14, 1, 35.0),
  (46, 1, 1, 150.0),
  (46, 13, 1, 90.0),
  (47, 1, 1, 150.0),
  (47, 2, 1, 80.0),
  (47, 6, 1, 104.5),
  (48, 1, 1, 150.0),
  (49, 1, 1, 142.5),
  (49, 2, 1, 80.0),
  (50, 1, 1, 135.0),
  (51, 1, 1, 150.0),
  (51, 14, 1, 35.0),
  (52, 1, 1, 150.0),
  (53, 3, 1, 220.0),
  (54, 3, 1, 220.0),
  (55, 1, 1, 150.0),
  (55, 13, 1, 81.0),
  (56, 3, 1, 220.0),
  (56, 7, 1, 75.0),
  (57, 1, 1, 150.0),
  (58, 1, 1, 135.0),
  (59, 3, 1, 209.0),
  (59, 14, 1, 33.25),
  (60, 1, 1, 150.0),
  (60, 2, 1, 76.0),
  (61, 3, 1, 220.0),
  (62, 3, 1, 220.0),
  (63, 1, 1, 150.0),
  (64, 3, 1, 220.0),
  (65, 1, 1, 150.0),
  (66, 1, 1, 150.0),
  (66, 5, 1, 54.0),
  (66, 13, 1, 90.0),
  (67, 3, 1, 220.0),
  (67, 14, 2, 35.0),
  (68, 1, 1, 150.0),
  (68, 14, 1, 35.0),
  (69, 1, 1, 150.0),
  (69, 3, 1, 220.0),
  (70, 3, 1, 220.0),
  (70, 7, 1, 75.0),
  (71, 1, 1, 150.0),
  (71, 14, 2, 31.5),
  (72, 1, 1, 142.5),
  (73, 3, 1, 220.0),
  (73, 14, 1, 35.0),
  (74, 1, 1, 150.0),
  (74, 6, 1, 110.0),
  (75, 3, 1, 198.0),
  (76, 3, 1, 220.0),
  (76, 9, 1, 180.0),
  (76, 11, 1, 650.0),
  (77, 1, 1, 150.0),
  (78, 3, 1, 220.0),
  (78, 14, 1, 31.5),
  (79, 1, 1, 150.0),
  (79, 7, 1, 75.0),
  (79, 14, 1, 35.0),
  (80, 3, 1, 220.0),
  (81, 1, 1, 135.0),
  (81, 13, 1, 81.0),
  (82, 1, 1, 135.0),
  (83, 3, 1, 198.0),
  (83, 14, 2, 35.0),
  (84, 1, 1, 150.0),
  (85, 3, 1, 220.0),
  (85, 7, 1, 71.25),
  (85, 14, 1, 33.25),
  (86, 1, 1, 150.0),
  (86, 6, 1, 104.5),
  (87, 3, 1, 198.0),
  (88, 3, 1, 198.0),
  (88, 14, 1, 35.0),
  (89, 1, 1, 150.0),
  (90, 3, 1, 220.0),
  (91, 3, 1, 220.0),
  (91, 7, 1, 75.0),
  (91, 14, 2, 35.0),
  (92, 1, 1, 150.0),
  (93, 3, 1, 220.0),
  (93, 12, 1, 420.0),
  (93, 14, 1, 33.25),
  (94, 3, 1, 220.0),
  (94, 14, 1, 35.0),
  (95, 3, 1, 220.0),
  (96, 1, 1, 150.0),
  (96, 5, 1, 60.0),
  (97, 3, 1, 209.0),
  (98, 1, 1, 150.0),
  (98, 3, 1, 220.0),
  (99, 2, 1, 80.0),
  (99, 3, 1, 220.0),
  (99, 7, 1, 75.0),
  (100, 3, 1, 220.0),
  (101, 3, 1, 198.0),
  (101, 12, 1, 420.0),
  (102, 1, 1, 150.0),
  (102, 6, 1, 110.0),
  (102, 13, 1, 90.0),
  (103, 1, 1, 135.0),
  (103, 5, 1, 60.0),
  (103, 14, 1, 35.0),
  (104, 1, 1, 142.5),
  (105, 1, 1, 142.5),
  (106, 3, 1, 220.0),
  (107, 1, 1, 142.5),
  (108, 1, 1, 142.5),
  (108, 7, 1, 75.0),
  (109, 3, 1, 220.0),
  (110, 3, 1, 220.0),
  (111, 3, 1, 220.0),
  (111, 7, 1, 75.0),
  (112, 1, 1, 142.5),
  (112, 14, 1, 35.0),
  (113, 3, 1, 220.0),
  (114, 1, 1, 150.0),
  (114, 13, 1, 90.0),
  (115, 1, 1, 150.0),
  (115, 7, 1, 71.25),
  (116, 1, 1, 150.0),
  (116, 3, 1, 220.0),
  (116, 14, 1, 35.0),
  (117, 3, 1, 220.0),
  (117, 7, 1, 75.0),
  (118, 1, 1, 150.0),
  (119, 1, 1, 150.0),
  (119, 13, 1, 85.5),
  (120, 1, 1, 150.0),
  (121, 1, 1, 150.0),
  (121, 6, 1, 110.0),
  (122, 3, 1, 220.0),
  (123, 3, 1, 220.0),
  (124, 1, 1, 150.0),
  (124, 7, 1, 75.0),
  (125, 1, 1, 150.0),
  (126, 1, 1, 150.0),
  (127, 1, 1, 150.0),
  (127, 6, 1, 110.0),
  (127, 13, 1, 90.0),
  (128, 1, 1, 150.0),
  (129, 1, 1, 142.5),
  (130, 1, 1, 150.0),
  (130, 14, 2, 35.0),
  (131, 1, 1, 150.0),
  (131, 5, 1, 57.0),
  (131, 14, 1, 35.0),
  (132, 3, 1, 220.0),
  (132, 7, 1, 71.25),
  (133, 1, 1, 150.0),
  (133, 13, 1, 90.0),
  (134, 3, 1, 220.0),
  (134, 7, 1, 67.5),
  (135, 3, 1, 220.0),
  (136, 3, 1, 209.0),
  (137, 1, 1, 150.0),
  (137, 7, 1, 71.25),
  (137, 13, 1, 90.0),
  (138, 1, 1, 142.5),
  (139, 3, 1, 220.0),
  (140, 3, 1, 220.0),
  (140, 7, 1, 67.5),
  (141, 1, 1, 142.5),
  (141, 5, 1, 60.0),
  (142, 3, 1, 220.0),
  (143, 1, 1, 135.0),
  (144, 1, 1, 150.0),
  (144, 4, 1, 95.0),
  (144, 13, 1, 81.0),
  (145, 3, 1, 220.0),
  (145, 7, 1, 75.0),
  (146, 1, 1, 150.0),
  (146, 3, 1, 220.0),
  (147, 3, 1, 198.0),
  (147, 7, 1, 67.5),
  (147, 14, 1, 31.5),
  (148, 1, 1, 150.0),
  (148, 2, 1, 72.0),
  (149, 1, 1, 150.0),
  (150, 1, 1, 142.5),
  (150, 3, 1, 220.0),
  (151, 3, 1, 198.0),
  (151, 14, 1, 35.0),
  (152, 3, 1, 220.0),
  (152, 7, 1, 75.0),
  (153, 3, 1, 198.0),
  (154, 3, 1, 220.0),
  (154, 12, 1, 420.0),
  (154, 14, 1, 33.25),
  (155, 3, 1, 198.0),
  (156, 1, 1, 150.0),
  (156, 7, 1, 67.5),
  (157, 3, 1, 209.0),
  (157, 7, 1, 75.0),
  (158, 1, 1, 150.0),
  (158, 7, 1, 67.5),
  (159, 1, 1, 135.0),
  (160, 3, 1, 220.0);

Writing inserts.sql


In [21]:
# Carga alternativa (somente em banco vazio): recria as tabelas e executa os INSERTs
# executar(open('schema.sql', encoding='utf-8').read())
# executar(open('inserts.sql', encoding='utf-8').read())
print('Arquivo inserts.sql disponível em', os.path.abspath('inserts.sql'))

Arquivo inserts.sql disponível em /content/inserts.sql


## 8. Testes das regras de integridade
Cada teste abaixo **tenta violar** uma regra; o esperado é que o PostgreSQL **recuse** a operação (ou, no caso do CASCADE, propague a exclusão). O teste de CASCADE é feito dentro de uma transação desfeita com `ROLLBACK`, para não alterar os dados.

In [22]:
testes = {
    'FK inexistente (tutor 999)':
        "INSERT INTO animal (nome, sexo, peso_kg, id_tutor, id_raca) VALUES ('Teste', 'M', 5, 999, 1)",
    'CHECK sexo inválido':
        "INSERT INTO animal (nome, sexo, peso_kg, id_tutor, id_raca) VALUES ('Teste', 'X', 5, 1, 1)",
    'UNIQUE e-mail duplicado':
        "INSERT INTO tutor (nome, cpf, email, id_cidade) VALUES ('Fulano', '000.000.000-00', 'ana.santos1@gmail.com', 1)",
    'ON DELETE RESTRICT (estado em uso)':
        "DELETE FROM estado WHERE sigla = 'SC'",
    'CHECK status inválido':
        "UPDATE atendimento SET status = 'Pendente' WHERE id_atendimento = 1",
}
for nome, sql in testes.items():
    try:
        executar(sql)
        print(f'❌ {nome}: operação foi aceita (não deveria)')
    except psycopg2.Error as e:
        print(f'✅ {nome}: bloqueado -> {e.diag.message_primary}')

# ON DELETE CASCADE: excluir o tutor 1 remove seus animais (e atendimentos) -- desfeito com ROLLBACK
with conn.cursor() as cur:
    cur.execute('BEGIN')
    cur.execute('SELECT COUNT(id_animal) FROM animal WHERE id_tutor = 1')
    antes = cur.fetchone()[0]
    cur.execute('DELETE FROM tutor WHERE id_tutor = 1')
    cur.execute('SELECT COUNT(id_animal) FROM animal WHERE id_tutor = 1')
    depois = cur.fetchone()[0]
    cur.execute('ROLLBACK')
print(f'✅ ON DELETE CASCADE: animais do tutor 1 antes = {antes}, depois da exclusão = {depois} (ROLLBACK aplicado)')

✅ FK inexistente (tutor 999): bloqueado -> insert or update on table "animal" violates foreign key constraint "fk_animal_tutor"
✅ CHECK sexo inválido: bloqueado -> new row for relation "animal" violates check constraint "animal_sexo_check"
✅ UNIQUE e-mail duplicado: bloqueado -> duplicate key value violates unique constraint "tutor_email_key"
✅ ON DELETE RESTRICT (estado em uso): bloqueado -> update or delete on table "estado" violates foreign key constraint "fk_cidade_estado" on table "cidade"
✅ CHECK status inválido: bloqueado -> new row for relation "atendimento" violates check constraint "atendimento_status_check"
✅ ON DELETE CASCADE: animais do tutor 1 antes = 1, depois da exclusão = 0 (ROLLBACK aplicado)


## 9. Consultas SQL

Todas as consultas seguem as regras do desafio: **sem `*` na lista do SELECT** (colunas sempre nomeadas), **todas com cláusula `WHERE`** e **apenas uma tabela na cláusula `FROM`** (as demais entram via `JOIN … ON`).
> Observação: nas consultas 6, 9 e 10 o símbolo `*` aparece **apenas como operador aritmético de multiplicação** (`quantidade * valor_unitario`), nunca como “selecionar todas as colunas”.

### Matriz de cobertura dos conteúdos

| Consulta | Operadores | Agregação | ORDER BY | GROUP BY | HAVING | BETWEEN | IN | JOIN | LIKE |
|---|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 1 – Animais por faixa de peso | ✅ | | ✅ | | | | | | |
| 2 – Tutores com Gmail | ✅ | | ✅ | | | | | | ✅ |
| 3 – Vacinas e exames | ✅ | | ✅ | | | | ✅ | | |
| 4 – Agenda de março | | | ✅ | | | ✅ | | ✅ | |
| 5 – Perfil por espécie | ✅ | ✅ | ✅ | ✅ | | | | ✅ | |
| 6 – Faturamento por veterinário | ✅ | ✅ | ✅ | ✅ | ✅ | | | ✅ | |
| 7 – Preços por categoria | ✅ | ✅ | ✅ | ✅ | ✅ | | | | |
| 8 – Retrievers e SRD no Sul | ✅ | | ✅ | | | | ✅ | ✅ | ✅ |
| 9 – Clientes mais fiéis | ✅ | ✅ | ✅ | ✅ | ✅ | ✅ | | ✅ | |
| 10 – Serviços mais vendidos | ✅ | ✅ | ✅ | ✅ | ✅ | ✅ | | ✅ | |

### Consulta 1 — Animais de grande porte (fêmeas) e pets pequenos jovens

**Objetivo:** listar fêmeas com 20 kg ou mais (grupo com protocolo específico de anestesia/castração) **ou** animais com menos de 1 kg nascidos a partir de 2021 (aves e roedores jovens, que exigem manejo especial).

**Recursos:** operadores relacionais (`>=`, `<`, `>`, `=`), operadores lógicos (`AND`, `OR`) com parênteses para controlar a precedência, e `ORDER BY` com dois critérios.

**Explicação:**
* `WHERE (peso_kg >= 20 AND sexo = 'F') OR (peso_kg < 1 AND data_nascimento > '2020-12-31')` — os parênteses garantem que cada par de condições seja avaliado em conjunto antes do `OR`.
* `ORDER BY a.peso_kg DESC, a.nome ASC` — mais pesados primeiro; em caso de empate, ordem alfabética.

In [23]:
consulta("""
SELECT a.nome, a.sexo, a.peso_kg, a.data_nascimento
FROM animal a
WHERE (a.peso_kg >= 20 AND a.sexo = 'F')
   OR (a.peso_kg < 1 AND a.data_nascimento > '2020-12-31')
ORDER BY a.peso_kg DESC, a.nome ASC;
""")

10 linha(s) retornada(s)


,nome,sexo,peso_kg,data_nascimento
0,Julieta,F,43.97,2024-01-10
1,Estrela,F,41.49,2016-10-16
2,Zeus,F,37.03,2019-06-14
3,Cacau,F,36.24,2019-05-13
4,Belinha,F,34.95,2019-05-28
5,Princesa,F,33.71,2012-08-25
6,Ozzy,F,33.67,2018-09-26
7,Apolo,F,25.65,2023-08-14
8,Simba,M,0.12,2021-02-11
9,Mel,M,0.09,2024-05-29


### Consulta 2 — Tutores com e-mail Gmail e sobrenome/nome específicos

**Objetivo:** montar uma lista de contatos para uma campanha de e-mail marketing direcionada a tutores que usam Gmail e cujo sobrenome é *Silva* ou *Souza*, ou cujo nome começa com *Ma* (Mariana, Marcos, Marcelo, Matheus…).

**Recursos:** `LIKE` com curingas (`%` = qualquer sequência de caracteres), operadores lógicos `AND`/`OR` e `ORDER BY`.

**Explicação:**
* `email LIKE '%@gmail.com'` — termina com `@gmail.com`.
* `nome LIKE '%Silva'` / `'%Souza'` — **termina** com o sobrenome; `nome LIKE 'Ma%'` — **começa** com “Ma”.
* O resultado é ordenado pelo nome do tutor.

In [24]:
consulta("""
SELECT t.nome AS tutor, t.email, t.telefone
FROM tutor t
WHERE t.email LIKE '%@gmail.com'
  AND (t.nome LIKE '%Silva' OR t.nome LIKE '%Souza' OR t.nome LIKE 'Ma%')
ORDER BY t.nome;
""")

4 linha(s) retornada(s)


,tutor,email,telefone
0,Mariana Almeida,mariana.almeida13@gmail.com,(11) 95304-8019
1,Patrícia Souza,patricia.souza28@gmail.com,(47) 97596-6559
2,Paula Silva,paula.silva16@gmail.com,(21) 99797-5371
3,Sérgio Silva,sergio.silva19@gmail.com,(41) 99005-1319


### Consulta 3 — Vacinas e exames ativos, do mais caro para o mais barato

**Objetivo:** gerar a tabela de preços de vacinas e exames que a recepção apresenta ao cliente.

**Recursos:** operador `IN` (lista de valores), operador de comparação com booleano e `ORDER BY` composto.

**Explicação:**
* `categoria IN ('Vacina', 'Exame')` equivale a `categoria = 'Vacina' OR categoria = 'Exame'`, de forma mais legível.
* `ativo = TRUE` descarta serviços que a clínica não oferece mais.
* Ordena por categoria (A→Z) e, dentro de cada uma, pelo preço decrescente.

In [25]:
consulta("""
SELECT s.descricao, s.categoria, s.preco_base
FROM servico s
WHERE s.categoria IN ('Vacina', 'Exame')
  AND s.ativo = TRUE
ORDER BY s.categoria, s.preco_base DESC;
""")

7 linha(s) retornada(s)


,descricao,categoria,preco_base
0,Ultrassonografia abdominal,Exame,230.00
1,Raio-X,Exame,180.00
2,Eletrocardiograma,Exame,160.00
3,Hemograma completo,Exame,75.00
4,Vacina V4 felina,Vacina,110.00
5,Vacina V10 (cães),Vacina,95.00
6,Vacina antirrábica,Vacina,60.00


### Consulta 4 — Agenda de atendimentos de março/2026

**Objetivo:** relatório da agenda do mês de março, mostrando o paciente e o veterinário de cada atendimento.

**Recursos:** `BETWEEN` (intervalo de datas), `INNER JOIN` com 3 tabelas e `ORDER BY`.

**Explicação:**
* `FROM atendimento at` — apenas **uma** tabela no FROM; `animal` e `veterinario` são incluídas via `INNER JOIN … ON` usando as FKs.
* `data_hora BETWEEN '2026-03-01 00:00:00' AND '2026-03-31 23:59:59'` — o `BETWEEN` é inclusivo nas duas pontas; como a coluna é `TIMESTAMP`, o limite superior inclui o horário 23:59:59 para não perder os atendimentos do dia 31.
* Ordenado cronologicamente.

In [26]:
consulta("""
SELECT at.id_atendimento,
       at.data_hora,
       an.nome  AS animal,
       v.nome   AS veterinario,
       at.status
FROM atendimento at
INNER JOIN animal      an ON an.id_animal      = at.id_animal
INNER JOIN veterinario v  ON v.id_veterinario  = at.id_veterinario
WHERE at.data_hora BETWEEN '2026-03-01 00:00:00' AND '2026-03-31 23:59:59'
ORDER BY at.data_hora;
""")

16 linha(s) retornada(s)


,id_atendimento,data_hora,animal,veterinario,status
0,28,2026-03-02 14:30:00,Dora,Dr. Eduardo Prado,Concluído
1,29,2026-03-04 16:00:00,Apolo,Dra. Helena Duarte,Concluído
2,30,2026-03-06 16:30:00,Floquinho,Dr. Caio Batista,Concluído
3,31,2026-03-07 08:30:00,Marley,Dra. Sabrina Kowalski,Concluído
4,32,2026-03-09 11:00:00,Pandora,Dra. Viviane Lopes,Concluído
5,33,2026-03-09 12:00:00,Tieta,Dra. Lívia Moraes,Concluído
6,34,2026-03-10 13:00:00,Ozzy,Dr. Fábio Teixeira,Concluído
7,35,2026-03-12 13:00:00,Romeu,Dra. Lívia Moraes,Concluído
8,36,2026-03-18 09:00:00,Bidu,Dr. Eduardo Prado,Concluído
9,37,2026-03-20 10:30:00,Hulk,Dra. Helena Duarte,Concluído


### Consulta 5 — Perfil dos pacientes por espécie

**Objetivo:** saber quantos pacientes (nascidos a partir de 2014) a clínica atende por espécie e o peso médio, mínimo e máximo — útil para dimensionar estoque de medicamentos (dose por kg).

**Recursos:** funções de agregação (`COUNT`, `AVG`, `MIN`, `MAX`, `ROUND`), `GROUP BY`, dois `JOIN`s e `ORDER BY` por coluna agregada.

**Explicação:**
* `animal → raca → especie`: a espécie não fica no animal (3FN), então são necessários dois JOINs para chegar a ela.
* `WHERE data_nascimento >= '2014-01-01'` filtra as linhas **antes** do agrupamento.
* `GROUP BY e.nome` forma um grupo por espécie, e as funções de agregação são calculadas por grupo.
* `ORDER BY qtd_animais DESC` mostra primeiro as espécies mais atendidas.

In [27]:
consulta("""
SELECT e.nome              AS especie,
       COUNT(a.id_animal)  AS qtd_animais,
       ROUND(AVG(a.peso_kg), 2) AS peso_medio_kg,
       MIN(a.peso_kg)      AS menor_peso,
       MAX(a.peso_kg)      AS maior_peso
FROM animal a
INNER JOIN raca    r ON r.id_raca    = a.id_raca
INNER JOIN especie e ON e.id_especie = r.id_especie
WHERE a.data_nascimento >= '2014-01-01'
GROUP BY e.nome
ORDER BY qtd_animais DESC;
""")

4 linha(s) retornada(s)


,especie,qtd_animais,peso_medio_kg,menor_peso,maior_peso
0,Cão,27,21.25,5.52,43.97
1,Gato,15,5.41,3.09,9.90
2,Ave,4,0.07,0.03,0.12
3,Roedor,3,0.16,0.13,0.18


### Consulta 6 — Faturamento por veterinário (acima de R$ 3.000)

**Objetivo:** identificar os profissionais que mais geraram receita com atendimentos concluídos, exibindo apenas quem faturou mais de R$ 3.000,00.

**Recursos:** 3 `JOIN`s, operador aritmético `*`, `SUM`, `COUNT(DISTINCT …)`, `GROUP BY`, `HAVING` e `ORDER BY`.

**Explicação:**
* `SUM(ats.quantidade * ats.valor_unitario)` calcula a receita (o total não é armazenado, pois é dado derivado).
* `COUNT(DISTINCT at.id_atendimento)` — o `DISTINCT` é necessário porque, após o JOIN com `atendimento_servico`, um atendimento com vários serviços aparece em várias linhas.
* `WHERE status = 'Concluído'` filtra **linhas** antes de agrupar; `HAVING SUM(...) > 3000` filtra **grupos** depois da agregação — essa é a diferença entre WHERE e HAVING.

In [28]:
consulta("""
SELECT v.nome AS veterinario,
       esp.nome AS especialidade,
       COUNT(DISTINCT at.id_atendimento)            AS atendimentos,
       SUM(ats.quantidade * ats.valor_unitario)      AS faturamento
FROM atendimento at
INNER JOIN veterinario         v   ON v.id_veterinario     = at.id_veterinario
INNER JOIN especialidade       esp ON esp.id_especialidade = v.id_especialidade
INNER JOIN atendimento_servico ats ON ats.id_atendimento   = at.id_atendimento
WHERE at.status = 'Concluído'
GROUP BY v.nome, esp.nome
HAVING SUM(ats.quantidade * ats.valor_unitario) > 3000
ORDER BY faturamento DESC;
""")

6 linha(s) retornada(s)


,veterinario,especialidade,atendimentos,faturamento
0,Dra. Lívia Moraes,Animais Silvestres e Exóticos,31,8768.00
1,Dra. Helena Duarte,Clínica Geral,32,6352.00
2,Dr. Caio Batista,Odontologia,13,4663.50
3,Dra. Viviane Lopes,Clínica Geral,24,4550.50
4,Dr. Eduardo Prado,Clínica Geral,21,3847.50
5,Dr. Fábio Teixeira,Cardiologia,11,3046.75


### Consulta 7 — Estatísticas de preço por categoria de serviço

**Objetivo:** comparar o preço médio, mínimo e máximo de cada categoria de serviço ativo, considerando apenas categorias com pelo menos 2 serviços.

**Recursos:** `COUNT`, `AVG`, `MIN`, `MAX`, `ROUND`, `GROUP BY`, `HAVING` e `ORDER BY` (sem JOIN, sobre uma única tabela).

**Explicação:**
* `WHERE ativo = TRUE` considera somente o catálogo vigente.
* `GROUP BY categoria` agrupa os serviços; `HAVING COUNT(id_servico) >= 2` elimina categorias com um único serviço (ex.: Estética), pois média de um só valor não é representativa.
* Ordenado pelo preço médio decrescente.

In [29]:
consulta("""
SELECT s.categoria,
       COUNT(s.id_servico)          AS qtd_servicos,
       ROUND(AVG(s.preco_base), 2)  AS preco_medio,
       MIN(s.preco_base)            AS menor_preco,
       MAX(s.preco_base)            AS maior_preco
FROM servico s
WHERE s.ativo = TRUE
GROUP BY s.categoria
HAVING COUNT(s.id_servico) >= 2
ORDER BY preco_medio DESC;
""")

4 linha(s) retornada(s)


,categoria,qtd_servicos,preco_medio,menor_preco,maior_preco
0,Cirurgia,2,535.00,420.00,650.00
1,Exame,4,161.25,75.00,230.00
2,Consulta,3,150.00,80.00,220.00
3,Vacina,3,88.33,60.00,110.00


### Consulta 8 — Tutores de SC e PR com cães Retriever ou animais SRD

**Objetivo:** público-alvo de uma campanha regional (Sul: SC e PR) sobre displasia (comum em Retrievers) e de adoção responsável (animais SRD — sem raça definida).

**Recursos:** 4 `JOIN`s encadeados, `IN`, `LIKE`, operadores lógicos e `ORDER BY` com 3 colunas.

**Explicação:**
* Caminho dos JOINs: `animal → tutor → cidade → estado` e `animal → raca`.
* `est.sigla IN ('SC', 'PR')` restringe aos dois estados.
* `r.nome LIKE '%Retriever%'` encontra *Labrador Retriever* e *Golden Retriever*; `r.nome LIKE 'SRD%'` encontra *SRD (Cão)* e *SRD (Gato)*. Os parênteses garantem que o `OR` não “escape” do filtro de estado.
* Ordenado por UF, cidade e tutor.

In [30]:
consulta("""
SELECT t.nome  AS tutor,
       c.nome  AS cidade,
       est.sigla AS uf,
       an.nome AS animal,
       r.nome  AS raca
FROM animal an
INNER JOIN tutor  t   ON t.id_tutor    = an.id_tutor
INNER JOIN cidade c   ON c.id_cidade   = t.id_cidade
INNER JOIN estado est ON est.id_estado = c.id_estado
INNER JOIN raca   r   ON r.id_raca     = an.id_raca
WHERE est.sigla IN ('SC', 'PR')
  AND (r.nome LIKE '%Retriever%' OR r.nome LIKE 'SRD%')
ORDER BY est.sigla, c.nome, t.nome;
""")

12 linha(s) retornada(s)


,tutor,cidade,uf,animal,raca
0,Paula Silva,Curitiba,PR,Bento,SRD (Cão)
1,Paula Silva,Curitiba,PR,Pandora,SRD (Gato)
2,Paula Silva,Curitiba,PR,Apolo,Golden Retriever
3,Sérgio Silva,Curitiba,PR,Bidu,SRD (Gato)
4,Daniel Silva,Londrina,PR,Bob,SRD (Gato)
5,Henrique Almeida,Londrina,PR,Bela,SRD (Cão)
6,Bianca Lima,Florianópolis,SC,Pretinha,Golden Retriever
7,Gabriela Ferreira,Florianópolis,SC,Max,Golden Retriever
8,Thiago Rodrigues,Florianópolis,SC,Ozzy,Golden Retriever
9,Camila Pereira,Itajaí,SC,Jade,SRD (Cão)


### Consulta 9 — Clientes mais fiéis do 1º semestre/2026

**Objetivo:** encontrar os tutores com 3 ou mais atendimentos concluídos no 1º semestre de 2026, com o total gasto e o ticket médio — base para um programa de fidelidade.

**Recursos:** `BETWEEN`, 3 `JOIN`s, operadores aritméticos (`*` e `/`), `SUM`, `COUNT(DISTINCT)`, `ROUND`, `GROUP BY`, `HAVING`, `ORDER BY`.

**Explicação:**
* `atendimento → animal → tutor` liga o atendimento ao cliente; `atendimento_servico` traz os valores.
* `BETWEEN '2026-01-01' AND '2026-06-30 23:59:59'` delimita o semestre.
* Ticket médio = receita total ÷ nº de atendimentos distintos.
* `GROUP BY t.id_tutor, t.nome` — agrupa pelo ID para não misturar tutores homônimos.
* `HAVING COUNT(DISTINCT at.id_atendimento) >= 3` mantém só os clientes recorrentes; ordenado pelo total gasto.

In [31]:
consulta("""
SELECT t.nome AS tutor,
       COUNT(DISTINCT at.id_atendimento)                       AS qtd_atendimentos,
       SUM(ats.quantidade * ats.valor_unitario)                AS total_gasto,
       ROUND(SUM(ats.quantidade * ats.valor_unitario)
             / COUNT(DISTINCT at.id_atendimento), 2)          AS ticket_medio
FROM atendimento at
INNER JOIN animal              an  ON an.id_animal       = at.id_animal
INNER JOIN tutor               t   ON t.id_tutor         = an.id_tutor
INNER JOIN atendimento_servico ats ON ats.id_atendimento = at.id_atendimento
WHERE at.status = 'Concluído'
  AND at.data_hora BETWEEN '2026-01-01' AND '2026-06-30 23:59:59'
GROUP BY t.id_tutor, t.nome
HAVING COUNT(DISTINCT at.id_atendimento) >= 3
ORDER BY total_gasto DESC;
""")

13 linha(s) retornada(s)


,tutor,qtd_atendimentos,total_gasto,ticket_medio
0,Thiago Rodrigues,9,2551.00,283.44
1,Paula Silva,9,1738.00,193.11
2,André Oliveira,5,1286.00,257.20
3,Renata Lima,6,1239.00,206.50
4,Carla Barbosa,3,1198.50,399.50
5,João Almeida,3,1063.25,354.42
6,Mariana Almeida,4,1003.00,250.75
7,Leonardo Costa,4,821.00,205.25
8,Gustavo Souza,3,815.00,271.67
9,Lucas Pereira,3,790.50,263.50


### Consulta 10 — Serviços mais vendidos no 2º semestre e desconto médio concedido

**Objetivo:** ranking dos serviços com 5 ou mais unidades vendidas no 2º semestre de 2026 (desconsiderando cancelados), comparando o preço de tabela com o preço realmente praticado.

**Recursos:** operador `<>` (diferente), `BETWEEN`, 2 `JOIN`s, operadores aritméticos (`-`, `*`, `/`), `SUM`, `AVG`, `ROUND`, `GROUP BY`, `HAVING`, `ORDER BY`.

**Explicação:**
* `status <> 'Cancelado'` inclui concluídos e agendados (receita prevista).
* `desconto_medio_pct = (preco_base − AVG(valor_unitario)) × 100 ÷ preco_base` — mostra quanto, em média, a clínica deixa de cobrar em relação à tabela.
* `preco_base` entra no `GROUP BY` porque é usado fora de função de agregação.
* `HAVING SUM(quantidade) >= 5` remove serviços pouco vendidos; ordenado pela receita.

In [32]:
consulta("""
SELECT s.descricao,
       s.preco_base,
       SUM(ats.quantidade)                                      AS qtd_vendida,
       ROUND(AVG(ats.valor_unitario), 2)                        AS preco_medio_praticado,
       ROUND((s.preco_base - AVG(ats.valor_unitario)) * 100.0
             / s.preco_base, 2)                                 AS desconto_medio_pct,
       SUM(ats.quantidade * ats.valor_unitario)                 AS receita
FROM atendimento_servico ats
INNER JOIN servico     s  ON s.id_servico      = ats.id_servico
INNER JOIN atendimento at ON at.id_atendimento = ats.id_atendimento
WHERE at.status <> 'Cancelado'
  AND at.data_hora BETWEEN '2026-07-01' AND '2026-12-31 23:59:59'
GROUP BY s.id_servico, s.descricao, s.preco_base
HAVING SUM(ats.quantidade) >= 5
ORDER BY receita DESC;
""")

4 linha(s) retornada(s)


,descricao,preco_base,qtd_vendida,preco_medio_praticado,desconto_medio_pct,receita
0,Consulta especializada,220.00,27,215.93,1.85,5830.00
1,Consulta clínica,150.00,31,146.61,2.26,4545.00
2,Hemograma completo,75.00,15,72.25,3.67,1083.75
3,Aplicação de medicamento,35.00,7,35.00,0.00,245.00


## 10. Conclusão

**Grupo P — Bruno Siqueira Parente.**

O projeto modelou o processo de atendimento de uma clínica veterinária, partindo de uma ficha não normalizada até chegar a **11 tabelas em 3FN**, implementadas no **PostgreSQL** com chaves primárias, chaves estrangeiras com regras `CASCADE`/`RESTRICT` e restrições de domínio (`CHECK`, `UNIQUE`, `NOT NULL`, `DEFAULT`). A carga foi feita a partir de arquivos **CSV** (via `COPY`) e **JSON** (via `INSERT`), e há também um script alternativo de `INSERT`s.

As 10 consultas mostram como o modelo responde a perguntas reais do negócio — agenda, perfil dos pacientes, faturamento por profissional, clientes mais fiéis, serviços mais vendidos e descontos concedidos — usando operadores, funções de agregação, `ORDER BY`, `GROUP BY`, `HAVING`, `BETWEEN`, `IN`, `JOIN` e `LIKE`.

> *Os dados de tutores, animais e atendimentos são fictícios, gerados apenas para fins acadêmicos.*

In [33]:
conn.close()
print('Conexão encerrada.')

Conexão encerrada.
